<a href="https://colab.research.google.com/github/hannaginther/ENGG680_2025_Fall/blob/main/Project/Model_Code/TEST2_CB_LGB_Preprocessingv2_encodedfeatures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**CatBoost Regressor to predict length of stay**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load df
import pandas as pd
df_encoded = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/preprocessing_v2/sparcs_encoded.feather')
df_normal = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/preprocessing_v2/sparcs_clean_v2.feather')

In [ ]:
!pip install catboost
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.5 MB/s eta 0:00:00


In [ ]:
# @title
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime

# set style for better looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# create directory for saving models and plots
import os
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

# Separate featuresand target
X = df.drop('length_of_stay', axis=1)
y = df['length_of_stay']

# Remove special characters that LightGBM doesn't like
X.columns = (X.columns
             .str.replace('/', '_', regex=False)
             .str.replace(' ', '_', regex=False)
             .str.replace('[', '_', regex=False)
             .str.replace(']', '_', regex=False)
             .str.replace('{', '_', regex=False)
             .str.replace('}', '_', regex=False)
             .str.replace(':', '_', regex=False)
             .str.replace('"', '', regex=False)
             .str.replace("'", '', regex=False)
             .str.replace(',', '_', regex=False)  # ADD THIS
             .str.replace('-', '_', regex=False)  # ADD THIS
             .str.replace('.', '_', regex=False)  # ADD THIS TOO
)

print("Cleaned column names:")
print(X.columns.tolist())

#Train test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")
print(f"Target statistics - Mean: {y.mean():.2f}, Std: {y.std():.2f}, Min: {y.min()}, Max: {y.max()}")

# ================ LIGHT GBM ================
print("\n" + "="*50)
print("Training LightGBM...")
print("="*50)

# Convert boolean to int for LGBM
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
bool_cols = X_train_lgb.select_dtypes(include='bool').columns
X_train_lgb[bool_cols] = X_train_lgb[bool_cols].astype(int)
X_test_lgb[bool_cols] = X_test_lgb[bool_cols].astype(int)

# Store training history
evals_result_lgb = {}

lgb_model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

# Train model (need to include early stopping)
lgb_model.fit(
    X_train_lgb, y_train,
    eval_set=[(X_train_lgb, y_train), (X_test_lgb, y_test)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),
        lgb.log_evaluation(100),
        lgb.record_evaluation(evals_result_lgb)
    ]
)

# Predictions
y_pred_lgb_train = lgb_model.predict(X_train_lgb)
y_pred_lgb_test = lgb_model.predict(X_test_lgb)

# Metrics
lgb_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_train))
lgb_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_test))
lgb_train_mae = mean_absolute_error(y_train, y_pred_lgb_train)
lgb_test_mae = mean_absolute_error(y_test, y_pred_lgb_test)
lgb_train_r2 = r2_score(y_train, y_pred_lgb_train)
lgb_test_r2 = r2_score(y_test, y_pred_lgb_test)


print("\nLightGBM Results:")
print(f"Train RMSE: {lgb_train_rmse:.4f} | Test RMSE: {lgb_test_rmse:.4f}")
print(f"Train MAE: {lgb_train_mae:.4f} | Test MAE: {lgb_test_mae:.4f}")
print(f"Train R²: {lgb_train_r2:.4f} | Test R²: {lgb_test_r2:.4f}")
print(f"\nOverfitting Check:")
print(f"RMSE Difference (Test - Train): {lgb_test_rmse - lgb_train_rmse:.4f}")
print(f"R² Difference (Train - Test): {lgb_train_r2 - lgb_test_r2:.4f}")

# Save LightGBM model
lgb_model.booster_.save_model('models/lightgbm_model.txt')
joblib.dump(lgb_model, 'models/lightgbm_model.pkl')
print("LightGBM model saved to 'models/lightgbm_model.pkl'.")

# ============== CatBoost Regressor ==============
# Initialize CatBoostRegressor
print("\n" + "="*50)
print("Training CatBoostRegressor...")
print("="*50)

catboost_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    task_type='GPU'
)

catboost_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100
    )

# Predictions
y_pred_cat_train = catboost_model.predict(X_train)
y_pred_cat_test = catboost_model.predict(X_test)

# Metrics
cat_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_train))
cat_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_test))
cat_train_mae = mean_absolute_error(y_train, y_pred_cat_train)
cat_test_mae = mean_absolute_error(y_test, y_pred_cat_test)
cat_train_r2 = r2_score(y_train, y_pred_cat_train)
cat_test_r2 = r2_score(y_test, y_pred_cat_test)

print("\nCatBoost Results:")
print(f"Train RMSE: {cat_train_rmse:.4f} | Test RMSE: {cat_test_rmse:.4f}")
print(f"Train MAE: {cat_train_mae:.4f} | Test MAE: {cat_test_mae:.4f}")
print(f"Train R²: {cat_train_r2:.4f} | Test R²: {cat_test_r2:.4f}")
print(f"\nOverfitting Check:")
print(f"RMSE Difference (Test - Train): {cat_test_rmse - cat_train_rmse:.4f}")
print(f"R² Difference (Train - Test): {cat_train_r2 - cat_test_r2:.4f}")

# Save model
catboost_model.save_model('models/catboost_model.cbm')
joblib.dump(catboost_model, 'models/catboost_model.pkl')
print("CatBoost model saved to 'models/catboost_model.pkl'.")

Cleaned column names:
['hospital_county', 'facility_id', 'age_group', 'zip_code', 'gender', 'admission_type', 'ccsr_dx_code', 'ccsr_px_code', 'apr_drg_code', 'apr_mdc_code', 'apr_mortality_risk', 'apr_med_surg_desc', 'emergency_dept_indicator', 'health_service_area_Capital_Adirondacks', 'health_service_area_Central_NY', 'health_service_area_Finger_Lakes', 'health_service_area_Hudson_Valley', 'health_service_area_Long_Island', 'health_service_area_New_York_City', 'health_service_area_Southern_Tier', 'health_service_area_Western_NY', 'ethnicity_Multi_ethnic', 'ethnicity_Not_Span_Hispanic', 'ethnicity_Spanish_Hispanic', 'race_Black_African_American', 'race_Multi_racial', 'race_Other_Race', 'race_White', 'apr_severity_code_1', 'apr_severity_code_2', 'apr_severity_code_3', 'apr_severity_code_4', 'Payment_Typology_1_Blue_Cross_Blue_Shield', 'Payment_Typology_1_Department_of_Corrections', 'Payment_Typology_1_Federal_State_Local_VA', 'Payment_Typology_1_Managed_Care__Unspecified', 'Payment_Typ

In [ ]:
# @title
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import r2_score

# ==================== VISUALIZATIONS ====================
print("\n" + "="*50)
print("Generating Visualizations...")
print("="*50)

# 1. Training History (Learning Curves)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LightGBM learning curve
axes[0].plot(evals_result_lgb['training']['rmse'], label='Train', linewidth=2)
axes[0].plot(evals_result_lgb['valid_1']['rmse'], label='Validation', linewidth=2)
axes[0].set_xlabel('Iteration', fontsize=12)
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('LightGBM Learning Curve', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# CatBoost learning curve
cat_train_scores = catboost_model.evals_result_['learn']['RMSE']
cat_valid_scores = catboost_model.evals_result_['validation']['RMSE']
axes[1].plot(cat_train_scores, label='Train', linewidth=2, color='orange')
axes[1].plot(cat_valid_scores, label='Validation', linewidth=2, color='red')
axes[1].set_xlabel('Iteration', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('CatBoost Learning Curve', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/01_learning_curves.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/01_learning_curves.png")
plt.close()

# 2. Model Comparison
comparison_df = pd.DataFrame({
    'Metric': ['Train RMSE', 'Test RMSE', 'Train MAE', 'Test MAE', 'Train R²', 'Test R²'],
    'LightGBM': [lgb_train_rmse, lgb_test_rmse, lgb_train_mae, lgb_test_mae, lgb_train_r2, lgb_test_r2],
    'CatBoost': [cat_train_rmse, cat_test_rmse, cat_train_mae, cat_test_mae, cat_train_r2, cat_test_r2]
})

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison_df))
width = 0.35
bars1 = ax.bar(x - width/2, comparison_df['LightGBM'], width, label='LightGBM', alpha=0.8)
bars2 = ax.bar(x + width/2, comparison_df['CatBoost'], width, label='CatBoost', alpha=0.8)
ax.set_xlabel('Metrics', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['Metric'], rotation=45, ha='right')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('plots/02_model_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/02_model_comparison.png")
plt.close()

# 3. Feature Importance (Top 20)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# LightGBM Feature Importance
lgb_importance = pd.DataFrame({
    'feature': X_train_lgb.columns,
    'importance': lgb_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

axes[0].barh(range(len(lgb_importance)), lgb_importance['importance'], alpha=0.8)
axes[0].set_yticks(range(len(lgb_importance)))
axes[0].set_yticklabels(lgb_importance['feature'])
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('LightGBM Top 20 Features', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# CatBoost Feature Importance
cat_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': catboost_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

axes[1].barh(range(len(cat_importance)), cat_importance['importance'], alpha=0.8, color='orange')
axes[1].set_yticks(range(len(cat_importance)))
axes[1].set_yticklabels(cat_importance['feature'])
axes[1].invert_yaxis()
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('CatBoost Top 20 Features', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('plots/03_feature_importance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/03_feature_importance.png")
plt.close()

# 4. Predicted vs Actual (scatter plots)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subsample for visualization (if dataset is too large)
sample_size = min(10000, len(y_test))
sample_idx = np.random.choice(len(y_test), sample_size, replace=False)

# LightGBM
axes[0].scatter(y_test.iloc[sample_idx], y_pred_lgb_test[sample_idx], alpha=0.5, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Length of Stay', fontsize=12)
axes[0].set_ylabel('Predicted Length of Stay', fontsize=12)
axes[0].set_title(f'LightGBM: Predicted vs Actual (R²={lgb_test_r2:.4f})', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# CatBoost
axes[1].scatter(y_test.iloc[sample_idx], y_pred_cat_test[sample_idx], alpha=0.5, s=10, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Length of Stay', fontsize=12)
axes[1].set_ylabel('Predicted Length of Stay', fontsize=12)
axes[1].set_title(f'CatBoost: Predicted vs Actual (R²={cat_test_r2:.4f})', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/04_predicted_vs_actual.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/04_predicted_vs_actual.png")
plt.close()

# 5. Residual plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# LightGBM residuals
lgb_residuals_test = y_test - y_pred_lgb_test

axes[0, 0].scatter(y_pred_lgb_test[sample_idx], lgb_residuals_test.iloc[sample_idx], alpha=0.5, s=10)
axes[0, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 0].set_xlabel('Predicted Values', fontsize=12)
axes[0, 0].set_ylabel('Residuals', fontsize=12)
axes[0, 0].set_title('LightGBM: Residual Plot', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].hist(lgb_residuals_test, bins=50, edgecolor='black', alpha=0.7)
axes[0, 1].set_xlabel('Residuals', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('LightGBM: Residual Distribution', fontsize=14, fontweight='bold')
axes[0, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[0, 1].grid(True, alpha=0.3)

# CatBoost residuals
cat_residuals_test = y_test - y_pred_cat_test

axes[1, 0].scatter(y_pred_cat_test[sample_idx], cat_residuals_test.iloc[sample_idx], alpha=0.5, s=10, color='orange')
axes[1, 0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1, 0].set_xlabel('Predicted Values', fontsize=12)
axes[1, 0].set_ylabel('Residuals', fontsize=12)
axes[1, 0].set_title('CatBoost: Residual Plot', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(cat_residuals_test, bins=50, edgecolor='black', alpha=0.7, color='orange')
axes[1, 1].set_xlabel('Residuals', fontsize=12)
axes[1, 1].set_ylabel('Frequency', fontsize=12)
axes[1, 1].set_title('CatBoost: Residual Distribution', fontsize=14, fontweight='bold')
axes[1, 1].axvline(x=0, color='r', linestyle='--', lw=2)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/05_residual_analysis.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/05_residual_analysis.png")
plt.close()

# 6. Error distribution by prediction range
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bin predictions and calculate mean absolute error
bins = np.percentile(y_test, [0, 25, 50, 75, 100])
bin_labels = ['Q1 (0-25%)', 'Q2 (25-50%)', 'Q3 (50-75%)', 'Q4 (75-100%)']

lgb_binned_errors = []
cat_binned_errors = []

for i in range(len(bins)-1):
    mask = (y_test >= bins[i]) & (y_test < bins[i+1])
    if i == len(bins)-2:  # Include upper bound in last bin
        mask = (y_test >= bins[i]) & (y_test <= bins[i+1])

    lgb_binned_errors.append(np.abs(lgb_residuals_test[mask]).mean())
    cat_binned_errors.append(np.abs(cat_residuals_test[mask]).mean())

x = np.arange(len(bin_labels))
width = 0.35

axes[0].bar(x - width/2, lgb_binned_errors, width, label='LightGBM', alpha=0.8)
axes[0].bar(x + width/2, cat_binned_errors, width, label='CatBoost', alpha=0.8, color='orange')
axes[0].set_xlabel('Prediction Quantile', fontsize=12)
axes[0].set_ylabel('Mean Absolute Error', fontsize=12)
axes[0].set_title('MAE by Prediction Range', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(bin_labels)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Error percentages
axes[1].bar(x - width/2, [e/y_test.mean()*100 for e in lgb_binned_errors], width, label='LightGBM', alpha=0.8)
axes[1].bar(x + width/2, [e/y_test.mean()*100 for e in cat_binned_errors], width, label='CatBoost', alpha=0.8, color='orange')
axes[1].set_xlabel('Prediction Quantile', fontsize=12)
axes[1].set_ylabel('MAE as % of Mean Target', fontsize=12)
axes[1].set_title('Relative Error by Prediction Range', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(bin_labels)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('plots/06_error_by_range.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/06_error_by_range.png")
plt.close()

# 7. Overfitting visualization
fig, ax = plt.subplots(figsize=(10, 6))

models = ['LightGBM', 'CatBoost']
train_scores = [lgb_train_r2, cat_train_r2]
test_scores = [lgb_test_r2, cat_test_r2]

x = np.arange(len(models))
width = 0.35

bars1 = ax.bar(x - width/2, train_scores, width, label='Train R²', alpha=0.8)
bars2 = ax.bar(x + width/2, test_scores, width, label='Test R²', alpha=0.8)

ax.set_ylabel('R² Score', fontsize=12)
ax.set_title('Overfitting Check: Train vs Test Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=10)

# Add difference annotations
for i, model in enumerate(models):
    diff = train_scores[i] - test_scores[i]
    ax.text(i, max(train_scores[i], test_scores[i]) + 0.02,
            f'Δ={diff:.4f}', ha='center', fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/07_overfitting_check.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/07_overfitting_check.png")
plt.close()

# ==================== Summary Report ====================
print("\n" + "="*50)
print("FINAL SUMMARY REPORT")
print("="*50)

summary = f"""
MODEL PERFORMANCE SUMMARY
{'='*50}

LightGBM:
  Train RMSE: {lgb_train_rmse:.4f} | Test RMSE: {lgb_test_rmse:.4f}
  Train MAE:  {lgb_train_mae:.4f} | Test MAE:  {lgb_test_mae:.4f}
  Train R²:   {lgb_train_r2:.4f} | Test R²:   {lgb_test_r2:.4f}

  Overfitting Metrics:
    RMSE Gap: {lgb_test_rmse - lgb_train_rmse:.4f} ({'Good' if lgb_test_rmse - lgb_train_rmse < 0.5 else 'Check for overfitting'})
    R² Gap:   {lgb_train_r2 - lgb_test_r2:.4f} ({'Good' if lgb_train_r2 - lgb_test_r2 < 0.05 else 'Check for overfitting'})

CatBoost:
  Train RMSE: {cat_train_rmse:.4f} | Test RMSE: {cat_test_rmse:.4f}
  Train MAE:  {cat_train_mae:.4f} | Test MAE:  {cat_test_mae:.4f}
  Train R²:   {cat_train_r2:.4f} | Test R²:   {cat_test_r2:.4f}

  Overfitting Metrics:
    RMSE Gap: {cat_test_rmse - cat_train_rmse:.4f} ({'Good' if cat_test_rmse - cat_train_rmse < 0.5 else 'Check for overfitting'})
    R² Gap:   {cat_train_r2 - cat_test_r2:.4f} ({'Good' if cat_train_r2 - cat_test_r2 < 0.05 else 'Check for overfitting'})

WINNER: {'LightGBM' if lgb_test_r2 > cat_test_r2 else 'CatBoost'} (Test R²: {max(lgb_test_r2, cat_test_r2):.4f})

FILES SAVED:
  Models:
    - models/lightgbm_model.pkl
    - models/catboost_model.pkl

  Plots:
    - plots/01_learning_curves.png
    - plots/02_model_comparison.png
    - plots/03_feature_importance.png
    - plots/04_predicted_vs_actual.png
    - plots/05_residual_analysis.png
    - plots/06_error_by_range.png
    - plots/07_overfitting_check.png

{'='*50}
"""

print(summary)

# Save summary to file
with open('models/model_summary.txt', 'w') as f:
    f.write(summary)
print("✓ Summary saved to 'models/model_summary.txt'")

print("\n🎉 All visualizations generated successfully!")


Generating Visualizations...
✓ Saved: plots/01_learning_curves.png
✓ Saved: plots/02_model_comparison.png
✓ Saved: plots/03_feature_importance.png
✓ Saved: plots/04_predicted_vs_actual.png
✓ Saved: plots/05_residual_analysis.png
✓ Saved: plots/06_error_by_range.png
✓ Saved: plots/07_overfitting_check.png

FINAL SUMMARY REPORT

MODEL PERFORMANCE SUMMARY

LightGBM:
  Train RMSE: 6.6707 | Test RMSE: 6.8380
  Train MAE:  3.1911 | Test MAE:  3.2198
  Train R²:   0.4789 | Test R²:   0.4593
  
  Overfitting Metrics:
    RMSE Gap: 0.1673 (Good)
    R² Gap:   0.0196 (Good)

CatBoost:
  Train RMSE: 6.9576 | Test RMSE: 7.0450
  Train MAE:  3.3316 | Test MAE:  3.3406
  Train R²:   0.4331 | Test R²:   0.4261
  
  Overfitting Metrics:
    RMSE Gap: 0.0874 (Good)
    R² Gap:   0.0070 (Good)

WINNER: LightGBM (Test R²: 0.4593)

FILES SAVED:
  Models:
    - models/lightgbm_model.pkl
    - models/catboost_model.pkl
  
  Plots:
    - plots/01_learning_curves.png
    - plots/02_model_comparison.png
    -

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 24.1 MB/s eta 0:00:00


In [ ]:
# @title
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Prepare data (same as before)
X = df.drop('length_of_stay', axis=1)
y = df['length_of_stay']

# Clean column names
X.columns = (X.columns
             .str.replace('/', '_', regex=False)
             .str.replace(' ', '_', regex=False)
             .str.replace('[', '_', regex=False)
             .str.replace(']', '_', regex=False)
             .str.replace('{', '_', regex=False)
             .str.replace('}', '_', regex=False)
             .str.replace(':', '_', regex=False)
             .str.replace('"', '', regex=False)
             .str.replace("'", '', regex=False)
             .str.replace(',', '_', regex=False)
             .str.replace('-', '_', regex=False)
             .str.replace('.', '_', regex=False)
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert boolean columns for LightGBM
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
bool_cols = X_train_lgb.select_dtypes(include='bool').columns
X_train_lgb[bool_cols] = X_train_lgb[bool_cols].astype(int)
X_test_lgb[bool_cols] = X_test_lgb[bool_cols].astype(int)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# ==================== LightGBM Hyperparameter Tuning ====================
print("\n" + "="*70)
print("TUNING LIGHTGBM HYPERPARAMETERS")
print("="*70)

def objective_lgb(trial):
    """Objective function for LightGBM hyperparameter tuning"""

    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'random_state': 42,
        'n_jobs': -1,

        # Hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'n_estimators': 1000
    }

    model = lgb.LGBMRegressor(**param)

    model.fit(
        X_train_lgb, y_train,
        eval_set=[(X_test_lgb, y_test)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    y_pred = model.predict(X_test_lgb)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse

# Run optimization for LightGBM
study_lgb = optuna.create_study(direction='minimize', study_name='lightgbm_tuning')
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

print("\n" + "="*70)
print("LIGHTGBM TUNING RESULTS")
print("="*70)
print(f"Best RMSE: {study_lgb.best_value:.4f}")
print(f"\nBest Parameters:")
for key, value in study_lgb.best_params.items():
    print(f"  {key}: {value}")

# Train final LightGBM model with best parameters
print("\nTraining final LightGBM model with best parameters...")
best_lgb_params = study_lgb.best_params.copy()
best_lgb_params.update({
    'objective': 'regression',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'n_estimators': 1000
})

evals_result_lgb = {}
lgb_model_tuned = lgb.LGBMRegressor(**best_lgb_params)
lgb_model_tuned.fit(
    X_train_lgb, y_train,
    eval_set=[(X_train_lgb, y_train), (X_test_lgb, y_test)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.record_evaluation(evals_result_lgb)
    ]
)

# Evaluate tuned LightGBM
y_pred_lgb_train = lgb_model_tuned.predict(X_train_lgb)
y_pred_lgb_test = lgb_model_tuned.predict(X_test_lgb)

lgb_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_train))
lgb_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_test))
lgb_tuned_train_mae = mean_absolute_error(y_train, y_pred_lgb_train)
lgb_tuned_test_mae = mean_absolute_error(y_test, y_pred_lgb_test)
lgb_tuned_train_r2 = r2_score(y_train, y_pred_lgb_train)
lgb_tuned_test_r2 = r2_score(y_test, y_pred_lgb_test)

print("\nTuned LightGBM Results:")
print(f"Train RMSE: {lgb_tuned_train_rmse:.4f} | Test RMSE: {lgb_tuned_test_rmse:.4f}")
print(f"Train MAE: {lgb_tuned_train_mae:.4f} | Test MAE: {lgb_tuned_test_mae:.4f}")
print(f"Train R²: {lgb_tuned_train_r2:.4f} | Test R²: {lgb_tuned_test_r2:.4f}")

# Save tuned LightGBM model
joblib.dump(lgb_model_tuned, 'models/lightgbm_model_tuned.pkl')
print("✓ Tuned LightGBM model saved to 'models/lightgbm_model_tuned.pkl'")

# ==================== CatBoost Hyperparameter Tuning ====================
print("\n" + "="*70)
print("TUNING CATBOOST HYPERPARAMETERS")
print("="*70)

def objective_cat(trial):
    """Objective function for CatBoost hyperparameter tuning"""

    param = {
        'iterations': 1000,
        'random_seed': 42,
        'verbose': 0,
        'early_stopping_rounds': 50,
        'task_type': 'GPU',

        # Hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
    }

    model = CatBoostRegressor(**param)

    model.fit(
        X_train, y_train,
        eval_set=(X_test, y_test),
        verbose=0
    )

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    return rmse

# Run optimization for CatBoost
study_cat = optuna.create_study(direction='minimize', study_name='catboost_tuning')
study_cat.optimize(objective_cat, n_trials=50, show_progress_bar=True)

print("\n" + "="*70)
print("CATBOOST TUNING RESULTS")
print("="*70)
print(f"Best RMSE: {study_cat.best_value:.4f}")
print(f"\nBest Parameters:")
for key, value in study_cat.best_params.items():
    print(f"  {key}: {value}")

# Train final CatBoost model with best parameters
print("\nTraining final CatBoost model with best parameters...")
best_cat_params = study_cat.best_params.copy()
best_cat_params.update({
    'iterations': 1000,
    'random_seed': 42,
    'verbose': 0,
    'early_stopping_rounds': 50,
    'task_type': 'GPU'
})

catboost_model_tuned = CatBoostRegressor(**best_cat_params)
catboost_model_tuned.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100
)

# Evaluate tuned CatBoost
y_pred_cat_train = catboost_model_tuned.predict(X_train)
y_pred_cat_test = catboost_model_tuned.predict(X_test)

cat_tuned_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_train))
cat_tuned_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_test))
cat_tuned_train_mae = mean_absolute_error(y_train, y_pred_cat_train)
cat_tuned_test_mae = mean_absolute_error(y_test, y_pred_cat_test)
cat_tuned_train_r2 = r2_score(y_train, y_pred_cat_train)
cat_tuned_test_r2 = r2_score(y_test, y_pred_cat_test)

print("\nTuned CatBoost Results:")
print(f"Train RMSE: {cat_tuned_train_rmse:.4f} | Test RMSE: {cat_tuned_test_rmse:.4f}")
print(f"Train MAE: {cat_tuned_train_mae:.4f} | Test MAE: {cat_tuned_test_mae:.4f}")
print(f"Train R²: {cat_tuned_train_r2:.4f} | Test R²: {cat_tuned_test_r2:.4f}")

# Save tuned CatBoost model
joblib.dump(catboost_model_tuned, 'models/catboost_model_tuned.pkl')
print("✓ Tuned CatBoost model saved to 'models/catboost_model_tuned.pkl'")

# ==================== Comparison ====================
print("\n" + "="*70)
print("BEFORE vs AFTER TUNING COMPARISON")
print("="*70)

comparison_df = pd.DataFrame({
    'Model': ['LightGBM (Before)', 'LightGBM (After)', 'CatBoost (Before)', 'CatBoost (After)'],
    'Test RMSE': [6.8380, lgb_tuned_test_rmse, 7.0450, cat_tuned_test_rmse],
    'Test MAE': [3.2198, lgb_tuned_test_mae, 3.3406, cat_tuned_test_mae],
    'Test R²': [0.4593, lgb_tuned_test_r2, 0.4261, cat_tuned_test_r2],
    'Improvement': [
        'Baseline',
        f"{((6.8380 - lgb_tuned_test_rmse) / 6.8380 * 100):.2f}%",
        'Baseline',
        f"{((7.0450 - cat_tuned_test_rmse) / 7.0450 * 100):.2f}%"
    ]
})

print(comparison_df.to_string(index=False))

# ==================== Visualizations ====================
print("\n" + "="*70)
print("GENERATING TUNING VISUALIZATIONS")
print("="*70)

# 1. Optimization history
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LightGBM optimization history
lgb_values = [trial.value for trial in study_lgb.trials]
axes[0].plot(lgb_values, linewidth=2)
axes[0].axhline(y=min(lgb_values), color='r', linestyle='--', label=f'Best: {min(lgb_values):.4f}')
axes[0].set_xlabel('Trial', fontsize=12)
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('LightGBM Optimization History', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# CatBoost optimization history
cat_values = [trial.value for trial in study_cat.trials]
axes[1].plot(cat_values, linewidth=2, color='orange')
axes[1].axhline(y=min(cat_values), color='r', linestyle='--', label=f'Best: {min(cat_values):.4f}')
axes[1].set_xlabel('Trial', fontsize=12)
axes[1].set_ylabel('RMSE', fontsize=12)
axes[1].set_title('CatBoost Optimization History', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/08_optimization_history.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/08_optimization_history.png")
plt.close()

# 2. Parameter importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LightGBM parameter importance
lgb_importances = optuna.importance.get_param_importances(study_lgb)
lgb_imp_df = pd.DataFrame(list(lgb_importances.items()), columns=['Parameter', 'Importance']).sort_values('Importance', ascending=True)

axes[0].barh(lgb_imp_df['Parameter'], lgb_imp_df['Importance'])
axes[0].set_xlabel('Importance', fontsize=12)
axes[0].set_title('LightGBM Parameter Importance', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='x')

# CatBoost parameter importance
cat_importances = optuna.importance.get_param_importances(study_cat)
cat_imp_df = pd.DataFrame(list(cat_importances.items()), columns=['Parameter', 'Importance']).sort_values('Importance', ascending=True)

axes[1].barh(cat_imp_df['Parameter'], cat_imp_df['Importance'], color='orange')
axes[1].set_xlabel('Importance', fontsize=12)
axes[1].set_title('CatBoost Parameter Importance', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('plots/09_parameter_importance.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/09_parameter_importance.png")
plt.close()

# 3. Before/After comparison
fig, ax = plt.subplots(figsize=(12, 6))

models = ['LightGBM\n(Before)', 'LightGBM\n(After)', 'CatBoost\n(Before)', 'CatBoost\n(After)']
rmse_values = [6.8380, lgb_tuned_test_rmse, 7.0450, cat_tuned_test_rmse]
colors = ['skyblue', 'blue', 'lightsalmon', 'orangered']

bars = ax.bar(models, rmse_values, color=colors, alpha=0.7)
ax.set_ylabel('Test RMSE', fontsize=12)
ax.set_title('Model Performance: Before vs After Tuning', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, rmse_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Add improvement percentage
    if i == 1:  # LightGBM after
        improvement = ((6.8380 - val) / 6.8380 * 100)
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                f'↓{improvement:.2f}%', ha='center', va='center', fontsize=10,
                color='green', fontweight='bold')
    elif i == 3:  # CatBoost after
        improvement = ((7.0450 - val) / 7.0450 * 100)
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                f'↓{improvement:.2f}%', ha='center', va='center', fontsize=10,
                color='green', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/10_before_after_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/10_before_after_comparison.png")
plt.close()

print("\n🎉 Hyperparameter tuning complete! All models and plots saved.")

[I 2025-12-04 03:15:41,863] A new study created in memory with name: lightgbm_tuning


Training set size: (2078162, 41)
Test set size: (519541, 41)

TUNING LIGHTGBM HYPERPARAMETERS


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-04 03:16:34,875] Trial 0 finished with value: 6.878931424447724 and parameters: {'learning_rate': 0.022579233526981973, 'num_leaves': 45, 'max_depth': 12, 'min_child_samples': 88, 'subsample': 0.9988137251914686, 'colsample_bytree': 0.6108109200680324, 'reg_alpha': 0.36301840129219487, 'reg_lambda': 2.0890003772545573, 'min_split_gain': 0.6598909203595685}. Best is trial 0 with value: 6.878931424447724.
[I 2025-12-04 03:17:13,030] Trial 1 finished with value: 6.934473240002422 and parameters: {'learning_rate': 0.02785335034629847, 'num_leaves': 27, 'max_depth': 8, 'min_child_samples': 98, 'subsample': 0.758804370223967, 'colsample_bytree': 0.9618719279661502, 'reg_alpha': 5.327872187314083e-08, 'reg_lambda': 0.0013463633560998427, 'min_split_gain': 0.6517324649833008}. Best is trial 0 with value: 6.878931424447724.
[I 2025-12-04 03:17:49,603] Trial 2 finished with value: 6.930476771142867 and parameters: {'learning_rate': 0.035934665111300584, 'num_leaves': 22, 'max_depth': 

[I 2025-12-04 03:54:06,074] A new study created in memory with name: catboost_tuning



Tuned LightGBM Results:
Train RMSE: 6.3171 | Test RMSE: 6.7365
Train MAE: 3.0233 | Test MAE: 3.1310
Train R²: 0.5327 | Test R²: 0.4753
✓ Tuned LightGBM model saved to 'models/lightgbm_model_tuned.pkl'

TUNING CATBOOST HYPERPARAMETERS


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-04 03:54:17,154] Trial 0 finished with value: 7.045992661485833 and parameters: {'learning_rate': 0.042713661036004665, 'depth': 10, 'l2_leaf_reg': 7.98331800927551, 'bagging_temperature': 0.7956006399660903, 'random_strength': 0.3406546583448278, 'border_count': 36, 'min_data_in_leaf': 34}. Best is trial 0 with value: 7.045992661485833.
[I 2025-12-04 03:54:25,899] Trial 1 finished with value: 6.957256576823155 and parameters: {'learning_rate': 0.03935986390971778, 'depth': 8, 'l2_leaf_reg': 6.039804123788655, 'bagging_temperature': 0.4552909389079919, 'random_strength': 1.6266561409834668, 'border_count': 192, 'min_data_in_leaf': 67}. Best is trial 1 with value: 6.957256576823155.
[I 2025-12-04 03:54:31,652] Trial 2 finished with value: 7.130783406278788 and parameters: {'learning_rate': 0.06457879262643922, 'depth': 4, 'l2_leaf_reg': 5.137743838557423, 'bagging_temperature': 0.3312538942071326, 'random_strength': 6.4656142968937536, 'border_count': 182, 'min_data_in_leaf':

In [ ]:
# @title
import optuna
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("\n" + "="*70)
print("LOG-TRANSFORMED TARGET HYPERPARAMETER TUNING")
print("="*70)

# Create log-transformed target
y_train_log = np.log1p(y_train)  # log1p handles log(1+x) to avoid log(0)
y_test_log = np.log1p(y_test)

print(f"Original target - Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"Log target - Mean: {y_train_log.mean():.2f}, Std: {y_train_log.std():.2f}")

# ==================== LightGBM with Log Target ====================
print("\n" + "="*70)
print("TUNING LIGHTGBM WITH LOG-TRANSFORMED TARGET")
print("="*70)

def objective_lgb_log(trial):
    """Objective function for LightGBM with log-transformed target"""

    param = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'random_state': 42,
        'n_jobs': -1,

        # Hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        'n_estimators': 1000
    }

    model = lgb.LGBMRegressor(**param)

    model.fit(
        X_train_lgb, y_train_log,
        eval_set=[(X_test_lgb, y_test_log)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # Predict in log space
    y_pred_log = model.predict(X_test_lgb)

    # Convert back to original scale
    y_pred_original = np.expm1(y_pred_log)  # expm1 is inverse of log1p
    y_test_original = y_test

    # Calculate RMSE on original scale
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))

    return rmse

# Run optimization for LightGBM with log target
study_lgb_log = optuna.create_study(direction='minimize', study_name='lightgbm_log_tuning')
study_lgb_log.optimize(objective_lgb_log, n_trials=50, show_progress_bar=True)

print("\n" + "="*70)
print("LIGHTGBM LOG-TARGET TUNING RESULTS")
print("="*70)
print(f"Best RMSE (original scale): {study_lgb_log.best_value:.4f}")
print(f"\nBest Parameters:")
for key, value in study_lgb_log.best_params.items():
    print(f"  {key}: {value}")

# Train final LightGBM model with log target and best parameters
print("\nTraining final LightGBM model with log target and best parameters...")
best_lgb_log_params = study_lgb_log.best_params.copy()
best_lgb_log_params.update({
    'objective': 'regression',
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1,
    'n_estimators': 1000
})

evals_result_lgb_log = {}
lgb_model_log = lgb.LGBMRegressor(**best_lgb_log_params)
lgb_model_log.fit(
    X_train_lgb, y_train_log,
    eval_set=[(X_train_lgb, y_train_log), (X_test_lgb, y_test_log)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.record_evaluation(evals_result_lgb_log)
    ]
)

# Predictions - convert back to original scale
y_pred_lgb_log_train = np.expm1(lgb_model_log.predict(X_train_lgb))
y_pred_lgb_log_test = np.expm1(lgb_model_log.predict(X_test_lgb))

# Metrics on original scale
lgb_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_log_train))
lgb_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_log_test))
lgb_log_train_mae = mean_absolute_error(y_train, y_pred_lgb_log_train)
lgb_log_test_mae = mean_absolute_error(y_test, y_pred_lgb_log_test)
lgb_log_train_r2 = r2_score(y_train, y_pred_lgb_log_train)
lgb_log_test_r2 = r2_score(y_test, y_pred_lgb_log_test)

print("\nLightGBM Log-Target Results (Original Scale):")
print(f"Train RMSE: {lgb_log_train_rmse:.4f} | Test RMSE: {lgb_log_test_rmse:.4f}")
print(f"Train MAE: {lgb_log_train_mae:.4f} | Test MAE: {lgb_log_test_mae:.4f}")
print(f"Train R²: {lgb_log_train_r2:.4f} | Test R²: {lgb_log_test_r2:.4f}")

# Save model
joblib.dump(lgb_model_log, 'models/lightgbm_model_log.pkl')
print("✓ Log-target LightGBM model saved to 'models/lightgbm_model_log.pkl'")

# ==================== CatBoost with Log Target ====================
print("\n" + "="*70)
print("TUNING CATBOOST WITH LOG-TRANSFORMED TARGET")
print("="*70)

def objective_cat_log(trial):
    """Objective function for CatBoost with log-transformed target"""

    param = {
        'iterations': 1000,
        'random_seed': 42,
        'verbose': 0,
        'early_stopping_rounds': 50,
        'task_type': 'GPU',

        # Hyperparameters to tune
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength': trial.suggest_float('random_strength', 0, 10),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 100),
    }

    model = CatBoostRegressor(**param)

    model.fit(
        X_train, y_train_log,
        eval_set=(X_test, y_test_log),
        verbose=0
    )

    # Predict in log space
    y_pred_log = model.predict(X_test)

    # Convert back to original scale
    y_pred_original = np.expm1(y_pred_log)
    y_test_original = y_test

    # Calculate RMSE on original scale
    rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))

    return rmse

# Run optimization for CatBoost with log target
study_cat_log = optuna.create_study(direction='minimize', study_name='catboost_log_tuning')
study_cat_log.optimize(objective_cat_log, n_trials=50, show_progress_bar=True)

print("\n" + "="*70)
print("CATBOOST LOG-TARGET TUNING RESULTS")
print("="*70)
print(f"Best RMSE (original scale): {study_cat_log.best_value:.4f}")
print(f"\nBest Parameters:")
for key, value in study_cat_log.best_params.items():
    print(f"  {key}: {value}")

# Train final CatBoost model with log target and best parameters
print("\nTraining final CatBoost model with log target and best parameters...")
best_cat_log_params = study_cat_log.best_params.copy()
best_cat_log_params.update({
    'iterations': 1000,
    'random_seed': 42,
    'verbose': 0,
    'early_stopping_rounds': 50,
    'task_type': 'GPU'
})

catboost_model_log = CatBoostRegressor(**best_cat_log_params)
catboost_model_log.fit(
    X_train, y_train_log,
    eval_set=(X_test, y_test_log),
    verbose=100
)

# Predictions - convert back to original scale
y_pred_cat_log_train = np.expm1(catboost_model_log.predict(X_train))
y_pred_cat_log_test = np.expm1(catboost_model_log.predict(X_test))

# Metrics on original scale
cat_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_log_train))
cat_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_log_test))
cat_log_train_mae = mean_absolute_error(y_train, y_pred_cat_log_train)
cat_log_test_mae = mean_absolute_error(y_test, y_pred_cat_log_test)
cat_log_train_r2 = r2_score(y_train, y_pred_cat_log_train)
cat_log_test_r2 = r2_score(y_test, y_pred_cat_log_test)

print("\nCatBoost Log-Target Results (Original Scale):")
print(f"Train RMSE: {cat_log_train_rmse:.4f} | Test RMSE: {cat_log_test_rmse:.4f}")
print(f"Train MAE: {cat_log_train_mae:.4f} | Test MAE: {cat_log_test_mae:.4f}")
print(f"Train R²: {cat_log_train_r2:.4f} | Test R²: {cat_log_test_r2:.4f}")

# Save model
joblib.dump(catboost_model_log, 'models/catboost_model_log.pkl')
print("✓ Log-target CatBoost model saved to 'models/catboost_model_log.pkl'")

# ==================== Comparison: Original vs Log-Transformed ====================
print("\n" + "="*70)
print("ORIGINAL TARGET vs LOG-TRANSFORMED TARGET COMPARISON")
print("="*70)

comparison_log_df = pd.DataFrame({
    'Model': ['LightGBM (Original)', 'LightGBM (Log)', 'CatBoost (Original)', 'CatBoost (Log)'],
    'Test RMSE': [lgb_test_rmse, lgb_log_test_rmse, cat_test_rmse, cat_log_test_rmse],
    'Test MAE': [lgb_test_mae, lgb_log_test_mae, cat_test_mae, cat_log_test_mae],
    'Test R²': [lgb_test_r2, lgb_log_test_r2, cat_test_r2, cat_log_test_r2],
    'Improvement RMSE': [
        'Baseline',
        f"{((lgb_test_rmse - lgb_log_test_rmse) / lgb_test_rmse * 100):.2f}%",
        'Baseline',
        f"{((cat_test_rmse - cat_log_test_rmse) / cat_test_rmse * 100):.2f}%"
    ]
})

print(comparison_log_df.to_string(index=False))

# ==================== Visualizations for Log Models ====================
print("\n" + "="*70)
print("GENERATING LOG-TARGET VISUALIZATIONS")
print("="*70)

# 1. Comparison: Original vs Log
fig, ax = plt.subplots(figsize=(12, 6))

models = ['LightGBM\n(Original)', 'LightGBM\n(Log)', 'CatBoost\n(Original)', 'CatBoost\n(Log)']
rmse_values = [lgb_test_rmse, lgb_log_test_rmse, cat_test_rmse, cat_log_test_rmse]
colors = ['skyblue', 'blue', 'lightsalmon', 'orangered']

bars = ax.bar(models, rmse_values, color=colors, alpha=0.7)
ax.set_ylabel('Test RMSE', fontsize=12)
ax.set_title('Model Performance: Original vs Log-Transformed Target', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for i, (bar, val) in enumerate(zip(bars, rmse_values)):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Add improvement percentage
    if i == 1:  # LightGBM log
        improvement = ((lgb_test_rmse - val) / lgb_test_rmse * 100)
        color = 'green' if improvement > 0 else 'red'
        symbol = '↓' if improvement > 0 else '↑'
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                f'{symbol}{abs(improvement):.2f}%', ha='center', va='center', fontsize=10,
                color=color, fontweight='bold')
    elif i == 3:  # CatBoost log
        improvement = ((cat_test_rmse - val) / cat_test_rmse * 100)
        color = 'green' if improvement > 0 else 'red'
        symbol = '↓' if improvement > 0 else '↑'
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                f'{symbol}{abs(improvement):.2f}%', ha='center', va='center', fontsize=10,
                color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/08_original_vs_log_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/08_original_vs_log_comparison.png")
plt.close()

# 2. Predicted vs Actual for Log models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sample_size = min(10000, len(y_test))
sample_idx = np.random.choice(len(y_test), sample_size, replace=False)

# LightGBM Log
axes[0].scatter(y_test.iloc[sample_idx], y_pred_lgb_log_test[sample_idx], alpha=0.5, s=10)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Length of Stay', fontsize=12)
axes[0].set_ylabel('Predicted Length of Stay', fontsize=12)
axes[0].set_title(f'LightGBM (Log): Predicted vs Actual (R²={lgb_log_test_r2:.4f})', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# CatBoost Log
axes[1].scatter(y_test.iloc[sample_idx], y_pred_cat_log_test[sample_idx], alpha=0.5, s=10, color='orange')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Length of Stay', fontsize=12)
axes[1].set_ylabel('Predicted Length of Stay', fontsize=12)
axes[1].set_title(f'CatBoost (Log): Predicted vs Actual (R²={cat_log_test_r2:.4f})', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/09_predicted_vs_actual_log.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/09_predicted_vs_actual_log.png")
plt.close()

# 3. R² Score Comparison
fig, ax = plt.subplots(figsize=(12, 6))

models_r2 = ['LightGBM\n(Original)', 'LightGBM\n(Log)', 'CatBoost\n(Original)', 'CatBoost\n(Log)']
r2_values = [lgb_test_r2, lgb_log_test_r2, cat_test_r2, cat_log_test_r2]

bars = ax.bar(models_r2, r2_values, color=colors, alpha=0.7)
ax.set_ylabel('Test R² Score', fontsize=12)
ax.set_title('R² Score Comparison: Original vs Log-Transformed Target', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([min(r2_values) - 0.05, max(r2_values) + 0.05])

# Add value labels
for bar, val in zip(bars, r2_values):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('plots/10_r2_comparison_log.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/10_r2_comparison_log.png")
plt.close()

print("\n🎉 Log-transformed target tuning and visualizations complete!")

# Final summary
print("\n" + "="*70)
print("FINAL MODEL RECOMMENDATION")
print("="*70)

all_models = {
    'LightGBM (Original)': lgb_test_rmse,
    'LightGBM (Log)': lgb_log_test_rmse,
    'CatBoost (Original)': cat_test_rmse,
    'CatBoost (Log)': cat_log_test_rmse
}

best_model = min(all_models, key=all_models.get)
best_rmse = all_models[best_model]

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Test RMSE: {best_rmse:.4f}")
print(f"\n   Full Results:")
for model, rmse in sorted(all_models.items(), key=lambda x: x[1]):
    print(f"   {model}: {rmse:.4f}")

[I 2025-12-04 04:02:54,558] A new study created in memory with name: lightgbm_log_tuning



LOG-TRANSFORMED TARGET HYPERPARAMETER TUNING
Original target - Mean: 6.23, Std: 9.24
Log target - Mean: 1.62, Std: 0.75

TUNING LIGHTGBM WITH LOG-TRANSFORMED TARGET


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-04 04:03:42,906] Trial 0 finished with value: 7.405684567916907 and parameters: {'learning_rate': 0.033868657794924616, 'num_leaves': 59, 'max_depth': 5, 'min_child_samples': 74, 'subsample': 0.9277262809461284, 'colsample_bytree': 0.8708583593175572, 'reg_alpha': 1.4698681677781118e-05, 'reg_lambda': 0.07355361316017515, 'min_split_gain': 0.23584111095957716}. Best is trial 0 with value: 7.405684567916907.
[I 2025-12-04 04:04:25,523] Trial 1 finished with value: 7.191314443075049 and parameters: {'learning_rate': 0.048515210912268944, 'num_leaves': 45, 'max_depth': 8, 'min_child_samples': 17, 'subsample': 0.6593729205076022, 'colsample_bytree': 0.8067729458057967, 'reg_alpha': 4.202631627823968e-07, 'reg_lambda': 4.885137112763414e-06, 'min_split_gain': 0.10303349494985459}. Best is trial 1 with value: 7.191314443075049.
[I 2025-12-04 04:05:24,108] Trial 2 finished with value: 7.542137093391155 and parameters: {'learning_rate': 0.010214363815069228, 'num_leaves': 42, 'max_d

[I 2025-12-04 04:40:10,798] A new study created in memory with name: catboost_log_tuning



LightGBM Log-Target Results (Original Scale):
Train RMSE: 6.8908 | Test RMSE: 7.0539
Train MAE: 2.9119 | Test MAE: 2.9568
Train R²: 0.4440 | Test R²: 0.4246
✓ Log-target LightGBM model saved to 'models/lightgbm_model_log.pkl'

TUNING CATBOOST WITH LOG-TRANSFORMED TARGET


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-12-04 04:40:17,146] Trial 0 finished with value: 7.88432275593662 and parameters: {'learning_rate': 0.011367373182756208, 'depth': 6, 'l2_leaf_reg': 7.678471296629993, 'bagging_temperature': 0.25565615104144823, 'random_strength': 3.818426542825443, 'border_count': 171, 'min_data_in_leaf': 80}. Best is trial 0 with value: 7.88432275593662.
[I 2025-12-04 04:40:25,789] Trial 1 finished with value: 7.4615279186956105 and parameters: {'learning_rate': 0.02906600686165242, 'depth': 8, 'l2_leaf_reg': 2.0960104484981494, 'bagging_temperature': 0.9139366440558728, 'random_strength': 0.25296781748068264, 'border_count': 111, 'min_data_in_leaf': 62}. Best is trial 1 with value: 7.4615279186956105.
[I 2025-12-04 04:40:34,704] Trial 2 finished with value: 7.282914629873322 and parameters: {'learning_rate': 0.06239638300639671, 'depth': 8, 'l2_leaf_reg': 5.533972703108867, 'bagging_temperature': 0.30727148560474726, 'random_strength': 3.8975538924143924, 'border_count': 171, 'min_data_in_le

In [ ]:
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
import os

# Create directories for saving models and plots
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)

print("\n" + "="*70)
print("TRAINING FINAL MODELS WITH BEST PARAMETERS")
print("="*70)

# Prepare data
X = df_encoded.drop('length_of_stay', axis=1)
y = df_encoded['length_of_stay']

# Clean column names
X.columns = (X.columns
             .str.replace('/', '_', regex=False)
             .str.replace(' ', '_', regex=False)
             .str.replace('[', '_', regex=False)
             .str.replace(']', '_', regex=False)
             .str.replace('{', '_', regex=False)
             .str.replace('}', '_', regex=False)
             .str.replace(':', '_', regex=False)
             .str.replace('"', '', regex=False)
             .str.replace("'", '', regex=False)
             .str.replace(',', '_', regex=False)
             .str.replace('-', '_', regex=False)
             .str.replace('.', '_', regex=False)
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert boolean columns for LightGBM
X_train_lgb = X_train.copy()
X_test_lgb = X_test.copy()
bool_cols = X_train_lgb.select_dtypes(include='bool').columns
X_train_lgb[bool_cols] = X_train_lgb[bool_cols].astype(int)
X_test_lgb[bool_cols] = X_test_lgb[bool_cols].astype(int)

# Create log-transformed targets
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test samples")

# ==================== MODEL 1: LightGBM - Raw Target ====================
print("\n" + "="*70)
print("MODEL 1: LightGBM - Raw Target (Best Parameters)")
print("="*70)

lgb_raw_params = {
    'objective': 'regression',
    'learning_rate': 0.09227865396330576,
    'num_leaves': 100,
    'max_depth': 11,
    'min_child_samples': 45,
    'subsample': 0.8841111726749229,
    'colsample_bytree': 0.8098016966383056,
    'reg_alpha': 4.869682987000682e-05,
    'reg_lambda': 0.003065914044151063,
    'min_split_gain': 0.6455364903593377,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

evals_lgb_raw = {}
lgb_model_raw = lgb.LGBMRegressor(**lgb_raw_params)
lgb_model_raw.fit(
    X_train_lgb, y_train,
    eval_set=[(X_train_lgb, y_train), (X_test_lgb, y_test)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.record_evaluation(evals_lgb_raw)
    ]
)

y_pred_lgb_raw_train = lgb_model_raw.predict(X_train_lgb)
y_pred_lgb_raw_test = lgb_model_raw.predict(X_test_lgb)

lgb_raw_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_raw_train))
lgb_raw_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_raw_test))
lgb_raw_train_mae = mean_absolute_error(y_train, y_pred_lgb_raw_train)
lgb_raw_test_mae = mean_absolute_error(y_test, y_pred_lgb_raw_test)
lgb_raw_train_r2 = r2_score(y_train, y_pred_lgb_raw_train)
lgb_raw_test_r2 = r2_score(y_test, y_pred_lgb_raw_test)

print(f"Train RMSE: {lgb_raw_train_rmse:.4f} | Test RMSE: {lgb_raw_test_rmse:.4f}")
print(f"Train MAE: {lgb_raw_train_mae:.4f} | Test MAE: {lgb_raw_test_mae:.4f}")
print(f"Train R²: {lgb_raw_train_r2:.4f} | Test R²: {lgb_raw_test_r2:.4f}")

joblib.dump(lgb_model_raw, 'models/lgb_raw_final.pkl')
print("✓ Saved: models/lgb_raw_final.pkl")

# ==================== MODEL 2: LightGBM - Log Target ====================
print("\n" + "="*70)
print("MODEL 2: LightGBM - Log Target (Best Parameters)")
print("="*70)

lgb_log_params = {
    'objective': 'regression',
    'learning_rate': 0.07523360061054472,
    'num_leaves': 90,
    'max_depth': 10,
    'min_child_samples': 23,
    'subsample': 0.6103368578601003,
    'colsample_bytree': 0.7214164522135984,
    'reg_alpha': 2.1072009734713224e-07,
    'reg_lambda': 7.631214699149285e-05,
    'min_split_gain': 0.14276861724051298,
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

evals_lgb_log = {}
lgb_model_log = lgb.LGBMRegressor(**lgb_log_params)
lgb_model_log.fit(
    X_train_lgb, y_train_log,
    eval_set=[(X_train_lgb, y_train_log), (X_test_lgb, y_test_log)],
    eval_metric='rmse',
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.record_evaluation(evals_lgb_log)
    ]
)

# Convert predictions back to original scale
y_pred_lgb_log_train = np.expm1(lgb_model_log.predict(X_train_lgb))
y_pred_lgb_log_test = np.expm1(lgb_model_log.predict(X_test_lgb))

lgb_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_log_train))
lgb_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_log_test))
lgb_log_train_mae = mean_absolute_error(y_train, y_pred_lgb_log_train)
lgb_log_test_mae = mean_absolute_error(y_test, y_pred_lgb_log_test)
lgb_log_train_r2 = r2_score(y_train, y_pred_lgb_log_train)
lgb_log_test_r2 = r2_score(y_test, y_pred_lgb_log_test)

print(f"Train RMSE: {lgb_log_train_rmse:.4f} | Test RMSE: {lgb_log_test_rmse:.4f}")
print(f"Train MAE: {lgb_log_train_mae:.4f} | Test MAE: {lgb_log_test_mae:.4f}")
print(f"Train R²: {lgb_log_train_r2:.4f} | Test R²: {lgb_log_test_r2:.4f}")

joblib.dump(lgb_model_log, 'models/lgb_log_final.pkl')
print("✓ Saved: models/lgb_log_final.pkl")

# ==================== MODEL 3: CatBoost - Raw Target ====================
print("\n" + "="*70)
print("MODEL 3: CatBoost - Raw Target (Best Parameters)")
print("="*70)

cat_raw_params = {
    'learning_rate': 0.0998081758908799,
    'depth': 10,
    'l2_leaf_reg': 9.836977214469572,
    'bagging_temperature': 0.07207856355202602,
    'random_strength': 3.3265459806676927,
    'border_count': 255,
    'min_data_in_leaf': 98,
    'iterations': 1000,
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 50,
    'task_type': 'GPU'
}

cat_model_raw = CatBoostRegressor(**cat_raw_params)
cat_model_raw.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    verbose=100
)

y_pred_cat_raw_train = cat_model_raw.predict(X_train)
y_pred_cat_raw_test = cat_model_raw.predict(X_test)

cat_raw_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_raw_train))
cat_raw_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_raw_test))
cat_raw_train_mae = mean_absolute_error(y_train, y_pred_cat_raw_train)
cat_raw_test_mae = mean_absolute_error(y_test, y_pred_cat_raw_test)
cat_raw_train_r2 = r2_score(y_train, y_pred_cat_raw_train)
cat_raw_test_r2 = r2_score(y_test, y_pred_cat_raw_test)

print(f"\nTrain RMSE: {cat_raw_train_rmse:.4f} | Test RMSE: {cat_raw_test_rmse:.4f}")
print(f"Train MAE: {cat_raw_train_mae:.4f} | Test MAE: {cat_raw_test_mae:.4f}")
print(f"Train R²: {cat_raw_train_r2:.4f} | Test R²: {cat_raw_test_r2:.4f}")

joblib.dump(cat_model_raw, 'models/cat_raw_final.pkl')
print("✓ Saved: models/cat_raw_final.pkl")

# ==================== MODEL 4: CatBoost - Log Target ====================
print("\n" + "="*70)
print("MODEL 4: CatBoost - Log Target (Best Parameters)")
print("="*70)

cat_log_params = {
    'learning_rate': 0.0991796315705968,
    'depth': 9,
    'l2_leaf_reg': 5.889903414458395,
    'bagging_temperature': 0.6602658535172587,
    'random_strength': 6.059249164753076,
    'border_count': 250,
    'min_data_in_leaf': 100,
    'iterations': 1000,
    'random_seed': 42,
    'verbose': 100,
    'early_stopping_rounds': 50,
    'task_type': 'GPU'
}

cat_model_log = CatBoostRegressor(**cat_log_params)
cat_model_log.fit(
    X_train, y_train_log,
    eval_set=(X_test, y_test_log),
    verbose=100
)

# Convert predictions back to original scale
y_pred_cat_log_train = np.expm1(cat_model_log.predict(X_train))
y_pred_cat_log_test = np.expm1(cat_model_log.predict(X_test))

cat_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_log_train))
cat_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_log_test))
cat_log_train_mae = mean_absolute_error(y_train, y_pred_cat_log_train)
cat_log_test_mae = mean_absolute_error(y_test, y_pred_cat_log_test)
cat_log_train_r2 = r2_score(y_train, y_pred_cat_log_train)
cat_log_test_r2 = r2_score(y_test, y_pred_cat_log_test)

print(f"\nTrain RMSE: {cat_log_train_rmse:.4f} | Test RMSE: {cat_log_test_rmse:.4f}")
print(f"Train MAE: {cat_log_train_mae:.4f} | Test MAE: {cat_log_test_mae:.4f}")
print(f"Train R²: {cat_log_train_r2:.4f} | Test R²: {cat_log_test_r2:.4f}")

joblib.dump(cat_model_log, 'models/cat_log_final.pkl')
print("✓ Saved: models/cat_log_final.pkl")

# ==================== COMPREHENSIVE VISUALIZATIONS ====================
print("\n" + "="*70)
print("GENERATING COMPREHENSIVE VISUALIZATIONS")
print("="*70)

# Create results summary
results_summary = pd.DataFrame({
    'Model': ['LightGBM (Raw)', 'LightGBM (Log)', 'CatBoost (Raw)', 'CatBoost (Log)'],
    'Train RMSE': [lgb_raw_train_rmse, lgb_log_train_rmse, cat_raw_train_rmse, cat_log_train_rmse],
    'Test RMSE': [lgb_raw_test_rmse, lgb_log_test_rmse, cat_raw_test_rmse, cat_log_test_rmse],
    'Train MAE': [lgb_raw_train_mae, lgb_log_train_mae, cat_raw_train_mae, cat_log_train_mae],
    'Test MAE': [lgb_raw_test_mae, lgb_log_test_mae, cat_raw_test_mae, cat_log_test_mae],
    'Train R²': [lgb_raw_train_r2, lgb_log_train_r2, cat_raw_train_r2, cat_log_train_r2],
    'Test R²': [lgb_raw_test_r2, lgb_log_test_r2, cat_raw_test_r2, cat_log_test_r2]
})

print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)
print(results_summary.to_string(index=False))

# Save summary
results_summary.to_csv('models/final_results_summary.csv', index=False)
print("\n✓ Saved: models/final_results_summary.csv")

# 1. Overall Model Comparison - Test RMSE
fig, ax = plt.subplots(figsize=(12, 6))

models = ['LightGBM\n(Raw)', 'LightGBM\n(Log)', 'CatBoost\n(Raw)', 'CatBoost\n(Log)']
test_rmse_values = [lgb_raw_test_rmse, lgb_log_test_rmse, cat_raw_test_rmse, cat_log_test_rmse]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

bars = ax.bar(models, test_rmse_values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Test RMSE', fontsize=14, fontweight='bold')
ax.set_title('Final Model Comparison: Test RMSE', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([min(test_rmse_values) - 0.2, max(test_rmse_values) + 0.2])

# Add value labels and highlight best
best_idx = test_rmse_values.index(min(test_rmse_values))
for i, (bar, val) in enumerate(zip(bars, test_rmse_values)):
    height = bar.get_height()
    label_color = 'green' if i == best_idx else 'black'
    weight = 'bold' if i == best_idx else 'normal'
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12,
            fontweight=weight, color=label_color)
    if i == best_idx:
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                '⭐ BEST', ha='center', va='center', fontsize=14,
                fontweight='bold', color='gold')

plt.tight_layout()
plt.savefig('plots/final_01_overall_comparison.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_01_overall_comparison.png")
plt.close()

# 2. Train vs Test Performance - Check Overfitting
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE comparison
x = np.arange(len(models))
width = 0.35

train_rmse_values = [lgb_raw_train_rmse, lgb_log_train_rmse, cat_raw_train_rmse, cat_log_train_rmse]

bars1 = axes[0].bar(x - width/2, train_rmse_values, width, label='Train RMSE', alpha=0.8, color='skyblue')
bars2 = axes[0].bar(x + width/2, test_rmse_values, width, label='Test RMSE', alpha=0.8, color='coral')
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('Overfitting Check: Train vs Test RMSE', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Add gap labels
for i in range(len(models)):
    gap = test_rmse_values[i] - train_rmse_values[i]
    axes[0].text(i, max(train_rmse_values[i], test_rmse_values[i]) + 0.1,
                f'Δ={gap:.3f}', ha='center', fontsize=9, color='red', fontweight='bold')

# R² comparison
train_r2_values = [lgb_raw_train_r2, lgb_log_train_r2, cat_raw_train_r2, cat_log_train_r2]
test_r2_values = [lgb_raw_test_r2, lgb_log_test_r2, cat_raw_test_r2, cat_log_test_r2]

bars3 = axes[1].bar(x - width/2, train_r2_values, width, label='Train R²', alpha=0.8, color='lightgreen')
bars4 = axes[1].bar(x + width/2, test_r2_values, width, label='Test R²', alpha=0.8, color='salmon')
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('Overfitting Check: Train vs Test R²', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

# Add gap labels
for i in range(len(models)):
    gap = train_r2_values[i] - test_r2_values[i]
    axes[1].text(i, min(train_r2_values[i], test_r2_values[i]) - 0.02,
                f'Δ={gap:.3f}', ha='center', fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/final_02_overfitting_check.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_02_overfitting_check.png")
plt.close()

# 3. Metrics Heatmap
fig, ax = plt.subplots(figsize=(12, 6))

metrics_data = results_summary[['Train RMSE', 'Test RMSE', 'Train MAE', 'Test MAE', 'Train R²', 'Test R²']].values
im = ax.imshow(metrics_data, cmap='RdYlGn_r', aspect='auto')

ax.set_xticks(np.arange(6))
ax.set_yticks(np.arange(4))
ax.set_xticklabels(['Train RMSE', 'Test RMSE', 'Train MAE', 'Test MAE', 'Train R²', 'Test R²'])
ax.set_yticklabels(models)

# Add text annotations
for i in range(4):
    for j in range(6):
        text = ax.text(j, i, f'{metrics_data[i, j]:.3f}',
                       ha="center", va="center", color="black", fontweight='bold')

ax.set_title('All Metrics Heatmap', fontsize=16, fontweight='bold')
fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('plots/final_03_metrics_heatmap.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_03_metrics_heatmap.png")
plt.close()

# 4. Predicted vs Actual - All Models
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
sample_size = min(10000, len(y_test))
sample_idx = np.random.choice(len(y_test), sample_size, replace=False)

predictions = [
    (y_pred_lgb_raw_test, 'LightGBM (Raw)', lgb_raw_test_r2, '#3498db'),
    (y_pred_lgb_log_test, 'LightGBM (Log)', lgb_log_test_r2, '#2ecc71'),
    (y_pred_cat_raw_test, 'CatBoost (Raw)', cat_raw_test_r2, '#e74c3c'),
    (y_pred_cat_log_test, 'CatBoost (Log)', cat_log_test_r2, '#f39c12')
]

for idx, (y_pred, title, r2, color) in enumerate(predictions):
    ax = axes[idx // 2, idx % 2]
    ax.scatter(y_test.iloc[sample_idx], y_pred[sample_idx], alpha=0.4, s=15, color=color)
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Length of Stay', fontsize=11)
    ax.set_ylabel('Predicted Length of Stay', fontsize=11)
    ax.set_title(f'{title}: Predicted vs Actual (R²={r2:.4f})',
                fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/final_04_predicted_vs_actual_all.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_04_predicted_vs_actual_all.png")
plt.close()

# 5. Residual Analysis - All Models
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

residuals = [
    (y_test - y_pred_lgb_raw_test, 'LightGBM (Raw)', '#3498db'),
    (y_test - y_pred_lgb_log_test, 'LightGBM (Log)', '#2ecc71'),
    (y_test - y_pred_cat_raw_test, 'CatBoost (Raw)', '#e74c3c'),
    (y_test - y_pred_cat_log_test, 'CatBoost (Log)', '#f39c12')
]

for idx, (resid, title, color) in enumerate(residuals):
    # Scatter plot
    ax1 = axes[idx // 2, idx % 2]
    ax1.scatter(predictions[idx][0][sample_idx], resid.iloc[sample_idx],
               alpha=0.4, s=10, color=color)
    ax1.axhline(y=0, color='red', linestyle='--', lw=2)
    ax1.set_xlabel('Predicted Values', fontsize=10)
    ax1.set_ylabel('Residuals', fontsize=10)
    ax1.set_title(f'{title}: Residual Plot', fontsize=11, fontweight='bold')
    ax1.grid(True, alpha=0.3)

    # Histogram
    ax2 = axes[idx // 2, (idx % 2) + 2]
    ax2.hist(resid, bins=50, edgecolor='black', alpha=0.7, color=color)
    ax2.axvline(x=0, color='red', linestyle='--', lw=2)
    ax2.set_xlabel('Residuals', fontsize=10)
    ax2.set_ylabel('Frequency', fontsize=10)
    ax2.set_title(f'{title}: Residual Distribution', fontsize=11, fontweight='bold')
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/final_05_residual_analysis_all.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_05_residual_analysis_all.png")
plt.close()

# 6. Feature Importance Comparison
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

models_fi = [
    (lgb_model_raw, X_train_lgb.columns, 'LightGBM (Raw)', '#3498db'),
    (lgb_model_log, X_train_lgb.columns, 'LightGBM (Log)', '#2ecc71'),
    (cat_model_raw, X_train.columns, 'CatBoost (Raw)', '#e74c3c'),
    (cat_model_log, X_train.columns, 'CatBoost (Log)', '#f39c12')
]

for idx, (model, columns, title, color) in enumerate(models_fi):
    ax = axes[idx // 2, idx % 2]

    importance_df = pd.DataFrame({
        'feature': columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)

    ax.barh(range(len(importance_df)), importance_df['importance'], alpha=0.8, color=color)
    ax.set_yticks(range(len(importance_df)))
    ax.set_yticklabels(importance_df['feature'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title(f'{title}: Top 15 Features', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('plots/final_06_feature_importance_all.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_06_feature_importance_all.png")
plt.close()

# 7. Raw vs Log Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# LightGBM comparison
lgb_comparison = ['Raw Target', 'Log Target']
lgb_rmse = [lgb_raw_test_rmse, lgb_log_test_rmse]
lgb_colors = ['#3498db', '#2ecc71']

bars1 = axes[0].bar(lgb_comparison, lgb_rmse, color=lgb_colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Test RMSE', fontsize=12)
axes[0].set_title('LightGBM: Raw vs Log Target', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars1, lgb_rmse):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

improvement_lgb = ((lgb_raw_test_rmse - lgb_log_test_rmse) / lgb_raw_test_rmse * 100)
axes[0].text(0.5, max(lgb_rmse) * 0.9, f'Change: {improvement_lgb:+.2f}%',
            ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# CatBoost comparison
cat_comparison = ['Raw Target', 'Log Target']
cat_rmse = [cat_raw_test_rmse, cat_log_test_rmse]
cat_colors = ['#e74c3c', '#f39c12']

bars2 = axes[1].bar(cat_comparison, cat_rmse, color=cat_colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Test RMSE', fontsize=12)
axes[1].set_title('CatBoost: Raw vs Log Target', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars2, cat_rmse):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
improvement_cat = ((cat_raw_test_rmse - cat_log_test_rmse) / cat_raw_test_rmse * 100)
axes[1].text(0.5, max(cat_rmse) * 0.9, f'Change: {improvement_cat:+.2f}%',
ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig('plots/final_07_raw_vs_log.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_07_raw_vs_log.png")
plt.close()
#==================== FINAL SUMMARY ====================
print("\n" + "="*70)
print("🏆 FINAL MODEL RANKING (by Test RMSE)")
print("="*70)
ranking = results_summary.sort_values('Test RMSE')[['Model', 'Test RMSE', 'Test MAE', 'Test R²']]
for idx, row in ranking.iterrows():
  rank = list(ranking.index).index(idx) + 1
  medal = ['🥇', '🥈', '🥉', '4️⃣'][rank-1]
  print(f"{medal} {row['Model']}: RMSE={row['Test RMSE']:.4f}, MAE={row['Test MAE']:.4f}, R²={row['Test R²']:.4f}")
best_model = ranking.iloc[0]['Model']
best_rmse = ranking.iloc[0]['Test RMSE']
print(f"\n🎯 BEST OVERALL MODEL: {best_model}")
print(f"   Test RMSE: {best_rmse:.4f}")
#Save final summary report
summary_text = f"""
FINAL MODEL COMPARISON REPORT
Generated: {datetime.now().strftime('%Y-%m-%m-%d %H:%M:%S')}
DATASET INFORMATION:

Training samples: {X_train.shape[0]:,}
Test samples: {X_test.shape[0]:,}
Features: {X_train.shape[1]}
Target: Length of Stay (days)

MODEL PERFORMANCE SUMMARY:
{results_summary.to_string(index=False)}
RANKING (by Test RMSE):
{ranking.to_string(index=False)}
BEST MODEL: {best_model}

Test RMSE: {best_rmse:.4f}
Test MAE: {ranking.iloc[0]['Test MAE']:.4f}
Test R²: {ranking.iloc[0]['Test R²']:.4f}

OVERFITTING ANALYSIS:

LightGBM (Raw): RMSE Gap = {lgb_raw_test_rmse - lgb_raw_train_rmse:.4f}, R² Gap = {lgb_raw_train_r2 - lgb_raw_test_r2:.4f}
LightGBM (Log): RMSE Gap = {lgb_log_test_rmse - lgb_log_train_rmse:.4f}, R² Gap = {lgb_log_train_r2 - lgb_log_test_r2:.4f}
CatBoost (Raw): RMSE Gap = {cat_raw_test_rmse - cat_raw_train_rmse:.4f}, R² Gap = {cat_raw_train_r2 - cat_raw_test_r2:.4f}
CatBoost (Log): RMSE Gap = {cat_log_test_rmse - cat_log_train_rmse:.4f}, R² Gap = {cat_log_train_r2 - cat_log_test_r2:.4f}

RAW vs LOG TARGET COMPARISON:

LightGBM: {'Log better' if lgb_log_test_rmse < lgb_raw_test_rmse else 'Raw better'} (Δ = {abs(lgb_raw_test_rmse - lgb_log_test_rmse):.4f} RMSE)
CatBoost: {'Log better' if cat_log_test_rmse < cat_raw_test_rmse else 'Raw better'} (Δ = {abs(cat_raw_test_rmse - cat_log_test_rmse):.4f} RMSE)

SAVED MODELS:

models/lgb_raw_final.pkl
models/lgb_log_final.pkl
models/cat_raw_final.pkl
models/cat_log_final.pkl

SAVED VISUALIZATIONS:

plots/final_01_overall_comparison.png
plots/final_02_overfitting_check.png
plots/final_03_metrics_heatmap.png
plots/final_04_predicted_vs_actual_all.png
plots/final_05_residual_analysis_all.png
plots/final_06_feature_importance_all.png
plots/final_07_raw_vs_log.png

================================================================================
"""
with open('models/FINAL_REPORT.txt', 'w') as f:
  f.write(summary_text)
  print("\n✓ Saved: models/FINAL_REPORT.txt")
  print("\n" + "="*70)
  print("🎉 ALL MODELS TRAINED AND VISUALIZATIONS COMPLETE!")
  print("="*70)


TRAINING FINAL MODELS WITH BEST PARAMETERS
Dataset: 2078162 train, 519541 test samples

MODEL 1: LightGBM - Raw Target (Best Parameters)
Train RMSE: 6.3171 | Test RMSE: 6.7365
Train MAE: 3.0233 | Test MAE: 3.1310
Train R²: 0.5327 | Test R²: 0.4753
✓ Saved: models/lgb_raw_final.pkl

MODEL 2: LightGBM - Log Target (Best Parameters)
Train RMSE: 6.8908 | Test RMSE: 7.0539
Train MAE: 2.9119 | Test MAE: 2.9568
Train R²: 0.4440 | Test R²: 0.4246
✓ Saved: models/lgb_log_final.pkl

MODEL 3: CatBoost - Raw Target (Best Parameters)
0:	learn: 8.9517950	test: 9.0102958	best: 9.0102958 (0)	total: 22.8ms	remaining: 22.8s
100:	learn: 7.0008960	test: 7.0813929	best: 7.0813929 (100)	total: 1.08s	remaining: 9.66s
200:	learn: 6.8439340	test: 6.9476421	best: 6.9476421 (200)	total: 2.11s	remaining: 8.39s
300:	learn: 6.7529609	test: 6.8792914	best: 6.8792914 (300)	total: 3.14s	remaining: 7.28s
400:	learn: 6.6927424	test: 6.8400747	best: 6.8400747 (400)	total: 4.18s	remaining: 6.25s
500:	learn: 6.6468589	tes

/tmp/ipython-input-1006752921.py:300: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/ipython-input-1006752921.py:300: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()
/tmp/ipython-input-1006752921.py:301: UserWarning: Glyph 11088 (\N{WHITE MEDIUM STAR}) missing from font(s) DejaVu Sans.
  plt.savefig('plots/final_01_overall_comparison.png', dpi=300, bbox_inches='tight')


✓ Saved: plots/final_01_overall_comparison.png
✓ Saved: plots/final_02_overfitting_check.png
✓ Saved: plots/final_03_metrics_heatmap.png
✓ Saved: plots/final_04_predicted_vs_actual_all.png
✓ Saved: plots/final_05_residual_analysis_all.png
✓ Saved: plots/final_06_feature_importance_all.png
✓ Saved: plots/final_07_raw_vs_log.png

🏆 FINAL MODEL RANKING (by Test RMSE)
🥇 LightGBM (Raw): RMSE=6.7365, MAE=3.1310, R²=0.4753
🥈 CatBoost (Raw): RMSE=6.7573, MAE=3.1601, R²=0.4720
🥉 LightGBM (Log): RMSE=7.0539, MAE=2.9568, R²=0.4246
4️⃣ CatBoost (Log): RMSE=7.1028, MAE=2.9835, R²=0.4166

🎯 BEST OVERALL MODEL: LightGBM (Raw)
   Test RMSE: 6.7365

✓ Saved: models/FINAL_REPORT.txt

🎉 ALL MODELS TRAINED AND VISUALIZATIONS COMPLETE!


In [ ]:
# Create new features from existing ones
import pandas as pd
import numpy as np

# Age-related features
df_normal['age_group_numeric'] = df_normal['age_group'].map({
    '0-17': 8.5, '18-29': 23.5, '30-49': 39.5,
    '50-69': 59.5, '70 or Older': 80
})

# Severity indicators
df_normal['high_severity'] = (df_normal['apr_severity_code'].isin(['3', '4'])).astype(int)
df_normal['high_mortality_risk'] = (df_normal['apr_mortality_risk'].isin(['Major', 'Extreme'])).astype(int)

# Interaction features
df_normal['age_x_severity'] = df_normal['age_group_numeric'] * df_normal['apr_severity_code'].astype(int)
df_normal['emergency_x_severity'] = df_normal['emergency_dept_indicator'].astype(int) * df_normal['apr_severity_code'].astype(int)

# Geographic features
df_normal['urban_hospital'] = df_normal['health_service_area'].isin(['New York City', 'Long Island']).astype(int)

# Healthcare utilization
df_normal['had_procedure'] = (df_normal['ccsr_px_code'] != 'none').astype(int)

# Payment complexity (if multiple payment types exist)
# df_normal['payment_complexity'] = df_normal['Payment Typology 1'].nunique()

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split

native_df = df_normal.copy()

# Identify categorical columns
cat_features = [
    'health_service_area', 'hospital_county', 'facility_id',
    'age_group', 'zip_code', 'gender', 'race', 'ethnicity',
    'admission_type', 'ccsr_dx_code', 'ccsr_px_code',
    'apr_drg_code', 'apr_mdc_code', 'apr_severity_code',
    'apr_mortality_risk', 'apr_med_surg_desc', 'Payment Typology 1'
]


# Get categorical column indices
cat_feature_indices = [X_train.columns.get_loc(col) for col in cat_features if col in X_train.columns]

# Train CatBoost with native categorical handling
catboost_native = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    cat_features=cat_feature_indices,  # Specify categorical features
    random_seed=42,
    verbose=100
)

catboost_native.fit(X_train, y_train, eval_set=(X_test, y_test))

0:	learn: 9.0802235	test: 9.1399590	best: 9.1399590 (0)	total: 1.05s	remaining: 17m 27s
100:	learn: 6.9673085	test: 7.0264170	best: 7.0264170 (100)	total: 1m 50s	remaining: 16m 24s
200:	learn: 6.8550393	test: 6.9273969	best: 6.9273969 (200)	total: 3m 49s	remaining: 15m 12s
300:	learn: 6.7913709	test: 6.8783472	best: 6.8783472 (300)	total: 5m 49s	remaining: 13m 31s
400:	learn: 6.7434778	test: 6.8460654	best: 6.8460654 (400)	total: 7m 48s	remaining: 11m 39s
500:	learn: 6.7089161	test: 6.8253212	best: 6.8253212 (500)	total: 9m 47s	remaining: 9m 44s
600:	learn: 6.6804980	test: 6.8094715	best: 6.8094715 (600)	total: 11m 45s	remaining: 7m 48s
700:	learn: 6.6557146	test: 6.7979368	best: 6.7979368 (700)	total: 13m 47s	remaining: 5m 52s
800:	learn: 6.6339198	test: 6.7885610	best: 6.7885610 (800)	total: 15m 48s	remaining: 3m 55s
900:	learn: 6.6151548	test: 6.7804889	best: 6.7804889 (900)	total: 17m 48s	remaining: 1m 57s
999:	learn: 6.5985217	test: 6.7736424	best: 6.7736424 (999)	total: 19m 47s	r

In [ ]:
from category_encoders import TargetEncoder

highcard_df = df_normal.copy()

# High cardinality features that might benefit from target encoding
high_card_features = ['facility_id', 'zip_code', 'ccsr_dx_code', 'ccsr_px_code', 'apr_drg_code']

# Target encode
encoder = TargetEncoder(cols=high_card_features)
X_train_te = encoder.fit_transform(X_train[high_card_features], y_train)
X_test_te = encoder.transform(X_test[high_card_features])

# Combine with other features
X_train_combined = pd.concat([
    X_train.drop(high_card_features, axis=1),
    X_train_te
], axis=1)
X_test_combined = pd.concat([
    X_test.drop(high_card_features, axis=1),
    X_test_te
], axis=1)

ModuleNotFoundError: No module named 'category_encoders'

In [ ]:
# Try different transformations
from sklearn.preprocessing import PowerTransformer

# Square root transformation
y_train_sqrt = np.sqrt(y_train)
y_test_sqrt = np.sqrt(y_test)

# Box-Cox transformation
pt = PowerTransformer(method='box-cox')
y_train_boxcox = pt.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_boxcox = pt.transform(y_test.values.reshape(-1, 1)).ravel()

# Yeo-Johnson (handles zeros and negatives)
pt_yj = PowerTransformer(method='yeo-johnson')
y_train_yj = pt_yj.fit_transform(y_train.values.reshape(-1, 1)).ravel()
y_test_yj = pt_yj.transform(y_test.values.reshape(-1, 1)).ravel()

# Train models on each transformation
# Remember to inverse_transform predictions!

In [ ]:
# Weighted average ensemble
def ensemble_predict(X, weights=[0.3, 0.3, 0.2, 0.2]):
    pred_lgb_raw = lgb_model_raw.predict(X)
    pred_lgb_log = np.expm1(lgb_model_log.predict(X))
    pred_cat_raw = cat_model_raw.predict(X)
    pred_cat_log = np.expm1(cat_model_log.predict(X))

    return (weights[0] * pred_lgb_raw +
            weights[1] * pred_lgb_log +
            weights[2] * pred_cat_raw +
            weights[3] * pred_cat_log)

# Optimize weights
from scipy.optimize import minimize

def objective(weights):
    pred = ensemble_predict(X_test_lgb, weights)
    return mean_squared_error(y_test, pred)

result = minimize(objective, x0=[0.25, 0.25, 0.25, 0.25],
                  bounds=[(0, 1), (0, 1), (0, 1), (0, 1)],
                  constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1})

best_weights = result.x
print(f"Optimal weights: {best_weights}")

In [ ]:
# Create LOS bins for stratified sampling
df_normal['los_bin'] = pd.cut(df_normal['length_of_stay'],
                        bins=[0, 2, 5, 10, 20, np.inf],
                        labels=['very_short', 'short', 'medium', 'long', 'very_long'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df_normal['los_bin']  # Ensure balanced representation
)

In [ ]:
# SMOTE for regression (if you have very few long stays)
from imblearn.over_sampling import SMOTE

# Bin target for SMOTE
y_binned = pd.cut(y_train, bins=[0, 3, 7, 14, np.inf], labels=[0, 1, 2, 3])

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_binned)
# Get original continuous values back
y_train_balanced = y_train.iloc[y_train_balanced.index]

In [ ]:
import optuna

# Run all approaches and compare
approaches = {}

# 1. Current best (encoded)
approaches['Encoded_LGB_Raw'] = lgb_raw_test_rmse
approaches['Encoded_LGB_Log'] = lgb_log_test_rmse

# 2. Train with new features
# Add your feature engineering code here
# approaches['FeatureEng_LGB'] = train_and_evaluate(...)

# 3. Native categorical
# approaches['Native_CatBoost'] = train_and_evaluate(...)

# 4. Target encoding
# approaches['TargetEnc_LGB'] = train_and_evaluate(...)

# 5. Different transforms
# approaches['Sqrt_Transform'] = train_and_evaluate(...)

# 6. Ensemble
# approaches['Ensemble'] = ensemble_rmse

# 7. Two-stage
# approaches['TwoStage'] = train_and_evaluate(...)

# Compare all
comparison_df = pd.DataFrame({
    'Approach': list(approaches.keys()),
    'Test RMSE': list(approaches.values())
}).sort_values('Test RMSE')

print(comparison_df)

# Task
I will now begin by executing the first step of the plan.

First, I will modify the existing CatBoost training code to use the `df_normal` DataFrame, which includes the newly engineered features. I'll ensure that all categorical features are correctly identified and passed to CatBoost for native handling. After training, I will evaluate and record the performance metrics for this model.

Next, I will adapt the visualization code to incorporate the `catboost_native` model's performance alongside the previously trained models. This will involve updating the overall model comparison, overfitting checks, metrics heatmap, predicted vs. actual plots, residual analysis, and feature importance. All updated figures will be saved to the 'plots' directory.

Finally, I will revise the `FINAL_REPORT.txt` to include the performance metrics of the `catboost_native` model and provide an updated ranking of all models based on their test RMSE.

```python
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from datetime import datetime
import os

# Set style for better looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

# Create directories for saving models and plots
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)


# --- Data Preparation for CatBoost Native ---
print("\n" + "="*70)
print("PREPARING DATA FOR CATBOOST NATIVE MODEL")
print("="*70)

# Load df_normal (should already be loaded from previous cells but re-load for safety)
# df_normal = pd.read_feather('/content/drive/MyDrive/Sparcs_Datafiles/preprocessing_v2/sparcs_clean_v2.feather')

# Apply feature engineering defined in cell hQNK4_HWxXa-
# These lines are from cell hQNK4_HWxXa-
# Age-related features
df_normal['age_group_numeric'] = df_normal['age_group'].map({
    '0-17': 8.5, '18-29': 23.5, '30-49': 39.5,
    '50-69': 59.5, '70 or Older': 80
})

# Severity indicators
df_normal['high_severity'] = (df_normal['apr_severity_code'].isin(['3', '4'])).astype(int)
df_normal['high_mortality_risk'] = (df_normal['apr_mortality_risk'].isin(['Major', 'Extreme'])).astype(int)

# Interaction features
df_normal['age_x_severity'] = df_normal['age_group_numeric'] * df_normal['apr_severity_code'].astype(int)
df_normal['emergency_x_severity'] = df_normal['emergency_dept_indicator'].astype(int) * df_normal['apr_severity_code'].astype(int)

# Geographic features
df_normal['urban_hospital'] = df_normal['health_service_area'].isin(['New York City', 'Long Island']).astype(int)

# Healthcare utilization
df_normal['had_procedure'] = (df_normal['ccsr_px_code'] != 'none').astype(int)

# Separate features (X) and target (y)
X_native = df_normal.drop('length_of_stay', axis=1)
y_native = df_normal['length_of_stay']

# Identify categorical features for CatBoost native handling
# Re-evaluate all columns in X_native to find categorical types, including new engineered ones
categorical_features_native = X_native.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# CatBoost can also handle integer-encoded categoricals if explicitly told.
# For now, let's treat integer columns as numerical unless they are explicitly named categorical.
# This list needs to be updated with new features if they are categorical and not of object/category/bool dtype.
explicit_integer_categoricals = ['hospital_county', 'facility_id', 'zip_code', 'apr_drg_code', 'apr_mdc_code']

# Combine all identified categorical features
all_categorical_features_for_catboost = list(set(categorical_features_native + explicit_integer_categoricals))

# Ensure that the column names are compatible with CatBoostRegressor
X_native.columns = (X_native.columns
                    .str.replace('[^A-Za-z0-9_]+', '', regex=True) # Remove special characters
                    .str.replace(' ', '_', regex=False)
                   )

# Split data
X_train_native, X_test_native, y_train_native, y_test_native = train_test_split(
    X_native, y_native, test_size=0.2, random_state=42
)

# Get categorical column indices for CatBoost
cat_feature_indices = [X_train_native.columns.get_loc(col) for col in all_categorical_features_for_catboost if col in X_train_native.columns]

print(f"Features in X_train_native: {X_train_native.shape[1]}")
print(f"Identified native categorical features for CatBoost: {len(cat_feature_indices)} out of {X_train_native.shape[1]}")
print(f"Example categorical features: {np.array(X_train_native.columns)[cat_feature_indices[:5]]}")


# --- Train CatBoost with Native Categorical Features ---
print("\n" + "="*70)
print("TRAINING CATBOOST REGRESSOR WITH NATIVE CATEGORICAL FEATURES")
print("="*70)

catboost_native_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=3, # Default value for l2 regularization
    cat_features=cat_feature_indices,
    random_seed=42,
    verbose=100,
    early_stopping_rounds=50,
    task_type='GPU'
)

catboost_native_model.fit(
    X_train_native, y_train_native,
    eval_set=(X_test_native, y_test_native),
    verbose=100
)

# Predictions
y_pred_cat_native_train = catboost_native_model.predict(X_train_native)
y_pred_cat_native_test = catboost_native_model.predict(X_test_native)

# Metrics
cat_native_train_rmse = np.sqrt(mean_squared_error(y_train_native, y_pred_cat_native_train))
cat_native_test_rmse = np.sqrt(mean_squared_error(y_test_native, y_pred_cat_native_test))
cat_native_train_mae = mean_absolute_error(y_train_native, y_pred_cat_native_train)
cat_native_test_mae = mean_absolute_error(y_test_native, y_pred_cat_native_test)
cat_native_train_r2 = r2_score(y_train_native, y_pred_cat_native_train)
cat_native_test_r2 = r2_score(y_test_native, y_pred_cat_native_test)

print("\nCatBoost Native Results:")
print(f"Train RMSE: {cat_native_train_rmse:.4f} | Test RMSE: {cat_native_test_rmse:.4f}")
print(f"Train MAE: {cat_native_train_mae:.4f} | Test MAE: {cat_native_test_mae:.4f}")
print(f"Train R²: {cat_native_train_r2:.4f} | Test R²: {cat_native_test_r2:.4f}")

# Save model
joblib.dump(catboost_native_model, 'models/catboost_native_model.pkl')
print("✓ CatBoost native model saved to 'models/catboost_native_model.pkl'.")

# --- Update Model Comparison and Visualizations ---
print("\n" + "="*70)
print("UPDATING COMPREHENSIVE VISUALIZATIONS")
print("="*70)

# Re-evaluate all previously trained models for consistency
# Load previously saved models and re-calculate predictions/metrics if necessary
# Assuming `lgb_model_raw`, `lgb_model_log`, `cat_model_raw`, `cat_model_log` exist from previous steps or are loaded
# If they don't exist, load them from saved .pkl files
try:
    if 'lgb_model_raw' not in locals() or 'lgb_model_raw' not in globals():
        lgb_model_raw = joblib.load('models/lgb_raw_final.pkl')
    if 'lgb_model_log' not in locals() or 'lgb_model_log' not in globals():
        lgb_model_log = joblib.load('models/lgb_log_final.pkl')
    if 'cat_model_raw' not in locals() or 'cat_model_raw' not in globals():
        cat_model_raw = joblib.load('models/cat_raw_final.pkl')
    if 'cat_model_log' not in locals() or 'cat_model_log' not in globals():
        cat_model_log = joblib.load('models/cat_log_final.pkl')
except FileNotFoundError:
    print("Warning: Some previous models not found. Please ensure all previous training cells were run.")
    # Fallback to current kernel variables if files not found
    pass

# Ensure consistent X_test/y_test for predictions for comparison
# Note: For LightGBM encoded models, X_test_lgb should be used.
# For CatBoost encoded models, X_test (from df_encoded) should be used.
# For CatBoost native, X_test_native should be used.

# For consistent comparison, we need to ensure all models predict on the same X_test_native (with new features).
# This means re-preparing `X_test` and `X_train_lgb` from `df_normal` with new features for the old models too,
# which might change their performance slightly due to feature engineering.

# However, the plan specifically states to train CatBoost with Native Categorical Features using `df_normal` (including newly engineered features).
# The previous models were trained on `df_encoded`. To compare meaningfully, we should either:
# A) Re-train ALL models (LGBM raw/log, CatBoost raw/log) on `df_normal` + new features
# B) Keep the previous models as-is and acknowledge they are on a different feature set, comparing them to `catboost_native_model` as a new approach.
# Given the plan is to *update* the comparison, implying adding the new model, option B seems more appropriate
# unless explicit retraining of all models on `df_normal` is intended.

# For now, let's calculate predictions for existing models using their original X_test/X_test_lgb.
# The new catboost_native_model uses X_test_native and y_test_native which includes engineered features.
# It is important to remember this difference when interpreting results.

# Assuming the X_train, X_test, X_train_lgb, X_test_lgb variables were set from df_encoded in CblhMQFZt9fV.
# We will use these existing variables for the previously trained models to avoid re-training them here.

# Re-calculate predictions for existing models if they were loaded or for safety
y_pred_lgb_raw_train = lgb_model_raw.predict(X_train_lgb) if 'lgb_model_raw' in locals() else np.zeros_like(y_train)
y_pred_lgb_raw_test = lgb_model_raw.predict(X_test_lgb) if 'lgb_model_raw' in locals() else np.zeros_like(y_test)

y_pred_lgb_log_train = np.expm1(lgb_model_log.predict(X_train_lgb)) if 'lgb_model_log' in locals() else np.zeros_like(y_train)
y_pred_lgb_log_test = np.expm1(lgb_model_log.predict(X_test_lgb)) if 'lgb_model_log' in locals() else np.zeros_like(y_test)

y_pred_cat_raw_train = cat_model_raw.predict(X_train) if 'cat_model_raw' in locals() else np.zeros_like(y_train)
y_pred_cat_raw_test = cat_model_raw.predict(X_test) if 'cat_model_raw' in locals() else np.zeros_like(y_test)

y_pred_cat_log_train = np.expm1(cat_model_log.predict(X_train)) if 'cat_model_log' in locals() else np.zeros_like(y_train)
y_pred_cat_log_test = np.expm1(cat_model_log.predict(X_test)) if 'cat_model_log' in locals() else np.zeros_like(y_test)

# Recalculate metrics for previously trained models on their respective test sets (df_encoded)
lgb_raw_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_raw_train))
lgb_raw_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_raw_test))
lgb_raw_train_mae = mean_absolute_error(y_train, y_pred_lgb_raw_train)
lgb_raw_test_mae = mean_absolute_error(y_test, y_pred_lgb_raw_test)
lgb_raw_train_r2 = r2_score(y_train, y_pred_lgb_raw_train)
lgb_raw_test_r2 = r2_score(y_test, y_pred_lgb_raw_test)

lgb_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_lgb_log_train))
lgb_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_lgb_log_test))
lgb_log_train_mae = mean_absolute_error(y_train, y_pred_lgb_log_train)
lgb_log_test_mae = mean_absolute_error(y_test, y_pred_lgb_log_test)
lgb_log_train_r2 = r2_score(y_train, y_pred_lgb_log_train)
lgb_log_test_r2 = r2_score(y_test, y_pred_lgb_log_test)

cat_raw_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_raw_train))
cat_raw_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_raw_test))
cat_raw_train_mae = mean_absolute_error(y_train, y_pred_cat_raw_train)
cat_raw_test_mae = mean_absolute_error(y_test, y_pred_cat_raw_test)
cat_raw_train_r2 = r2_score(y_train, y_pred_cat_raw_train)
cat_raw_test_r2 = r2_score(y_test, y_pred_cat_raw_test)

cat_log_train_rmse = np.sqrt(mean_squared_error(y_train, y_pred_cat_log_train))
cat_log_test_rmse = np.sqrt(mean_squared_error(y_test, y_pred_cat_log_test))
cat_log_train_mae = mean_absolute_error(y_train, y_pred_cat_log_train)
cat_log_test_mae = mean_absolute_error(y_test, y_pred_cat_log_test)
cat_log_train_r2 = r2_score(y_train, y_pred_cat_log_train)
cat_log_test_r2 = r2_score(y_test, y_pred_cat_log_test)


# Create results summary including the new CatBoost native model
results_summary = pd.DataFrame({
    'Model': ['LightGBM (Raw)', 'LightGBM (Log)', 'CatBoost (Raw)', 'CatBoost (Log)', 'CatBoost (Native)'],
    'Train RMSE': [lgb_raw_train_rmse, lgb_log_train_rmse, cat_raw_train_rmse, cat_log_train_rmse, cat_native_train_rmse],
    'Test RMSE': [lgb_raw_test_rmse, lgb_log_test_rmse, cat_raw_test_rmse, cat_log_test_rmse, cat_native_test_rmse],
    'Train MAE': [lgb_raw_train_mae, lgb_log_train_mae, cat_raw_train_mae, cat_log_train_mae, cat_native_train_mae],
    'Test MAE': [lgb_raw_test_mae, lgb_log_test_mae, cat_raw_test_mae, cat_log_test_mae, cat_native_test_mae],
    'Train R²': [lgb_raw_train_r2, lgb_log_train_r2, cat_raw_train_r2, cat_log_train_r2, cat_native_train_r2],
    'Test R²': [lgb_raw_test_r2, lgb_log_test_r2, cat_raw_test_r2, cat_log_test_r2, cat_native_test_r2]
})

print("\n" + "="*70)
print("UPDATED FINAL RESULTS SUMMARY")
print("="*70)
print(results_summary.to_string(index=False))

# Save updated summary
results_summary.to_csv('models/final_results_summary.csv', index=False)
print("\n✓ Saved: models/final_results_summary.csv")

# 1. Overall Model Comparison - Test RMSE
fig, ax = plt.subplots(figsize=(14, 7))

models_labels = results_summary['Model'].tolist()
test_rmse_values = results_summary['Test RMSE'].tolist()
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'] # Added a new color for native CatBoost

bars = ax.bar(models_labels, test_rmse_values, color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
ax.set_ylabel('Test RMSE', fontsize=14, fontweight='bold')
ax.set_title('Final Model Comparison: Test RMSE', fontsize=16, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([min(test_rmse_values) * 0.95, max(test_rmse_values) * 1.05])

# Add value labels and highlight best
best_idx = test_rmse_values.index(min(test_rmse_values))
for i, (bar, val) in enumerate(zip(bars, test_rmse_values)):
    height = bar.get_height()
    label_color = 'green' if i == best_idx else 'black'
    weight = 'bold' if i == best_idx else 'normal'
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12,
            fontweight=weight, color=label_color)
    if i == best_idx:
        ax.text(bar.get_x() + bar.get_width()/2., height * 0.5,
                '⭐ BEST', ha='center', va='center', fontsize=14,
                fontweight='bold', color='gold')

plt.tight_layout()
plt.savefig('plots/final_01_overall_comparison_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_01_overall_comparison_v2.png")
plt.close()

# 2. Train vs Test Performance - Check Overfitting
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# RMSE comparison
x = np.arange(len(models_labels))
width = 0.35

train_rmse_values = results_summary['Train RMSE'].tolist()
test_rmse_values = results_summary['Test RMSE'].tolist()

bars1 = axes[0].bar(x - width/2, train_rmse_values, width, label='Train RMSE', alpha=0.8, color='skyblue')
bars2 = axes[0].bar(x + width/2, test_rmse_values, width, label='Test RMSE', alpha=0.8, color='coral')
axes[0].set_ylabel('RMSE', fontsize=12)
axes[0].set_title('Overfitting Check: Train vs Test RMSE', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models_labels, rotation=45, ha='right')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3, axis='y')

# Add gap labels
for i in range(len(models_labels)):
    gap = test_rmse_values[i] - train_rmse_values[i]
    axes[0].text(i, max(train_rmse_values[i], test_rmse_values[i]) + 0.1,
                f'Δ={gap:.3f}', ha='center', fontsize=9, color='red', fontweight='bold')

# R² comparison
train_r2_values = results_summary['Train R²'].tolist()
test_r2_values = results_summary['Test R²'].tolist()

bars3 = axes[1].bar(x - width/2, train_r2_values, width, label='Train R²', alpha=0.8, color='lightgreen')
bars4 = axes[1].bar(x + width/2, test_r2_values, width, label='Test R²', alpha=0.8, color='salmon')
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('Overfitting Check: Train vs Test R²', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models_labels, rotation=45, ha='right')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3, axis='y')

# Add gap labels
for i in range(len(models_labels)):
    gap = train_r2_values[i] - test_r2_values[i]
    axes[1].text(i, min(train_r2_values[i], test_r2_values[i]) - 0.02,
                f'Δ={gap:.3f}', ha='center', fontsize=9, color='red', fontweight='bold')

plt.tight_layout()
plt.savefig('plots/final_02_overfitting_check_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_02_overfitting_check_v2.png")
plt.close()

# 3. Metrics Heatmap
fig, ax = plt.subplots(figsize=(14, 7))

metrics_data = results_summary[['Train RMSE', 'Test RMSE', 'Train MAE', 'Test MAE', 'Train R²', 'Test R²']].values
im = ax.imshow(metrics_data, cmap='RdYlGn_r', aspect='auto')

ax.set_xticks(np.arange(6))
ax.set_yticks(np.arange(len(models_labels)))
ax.set_xticklabels(['Train RMSE', 'Test RMSE', 'Train MAE', 'Test MAE', 'Train R²', 'Test R²'], rotation=45, ha='right')
ax.set_yticklabels(models_labels)

# Add text annotations
for i in range(len(models_labels)):
    for j in range(6):
        text = ax.text(j, i, f'{metrics_data[i, j]:.3f}',
                       ha="center", va="center", color="black", fontweight='bold')

ax.set_title('All Metrics Heatmap', fontsize=16, fontweight='bold')
fig.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig('plots/final_03_metrics_heatmap_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_03_metrics_heatmap_v2.png")
plt.close()

# 4. Predicted vs Actual - All Models
fig, axes = plt.subplots(3, 2, figsize=(18, 20)) # Changed to 3 rows to accommodate new model
sample_size = min(10000, len(y_test_native)) # Use y_test_native for consistent sampling
sample_idx = np.random.choice(len(y_test_native), sample_size, replace=False)

# Need to make sure X_test_lgb/X_test are aligned with y_test/y_test_native
# Assuming X_test_lgb and X_test were split from df_encoded and y_test from df_encoded
# and X_test_native was split from df_normal and y_test_native from df_normal
# For this plot, using y_test_native as the actual target and predicting on X_test_native for all models
# This means we need to predict the old models on the new feature set `X_test_native` if possible,
# or clearly state the difference in data used.
# Given the previous note, let's keep the old models predicting on their original test sets (`X_test_lgb`/`X_test`)
# and use y_test for them, and use X_test_native/y_test_native for the new CatBoost native model.
# This makes the "Predicted vs Actual - All Models" plot a bit tricky to compare directly.

# A better approach for this specific plot:
# 1. Use y_test_native as the actual values for ALL predictions.
# 2. Ensure all models can predict on X_test_native. This might require handling one-hot encoded columns for native CatBoost model.
#    Since X_test_native has non-encoded categoricals, the LGBM and old CatBoost models trained on one-hot encoded data (df_encoded)
#    might not be able to predict directly on it.

# To simplify and ensure comparability *within this plot*, let's predict all models on X_test_native.
# This implies re-encoding X_test_native for LGBM and original CatBoost models. This is complex and might not be what's intended.

# Let's revert to a simpler interpretation for this plot:
# plot for LightGBM models uses y_test and its predictions on X_test_lgb (from df_encoded)
# plot for CatBoost encoded models uses y_test and its predictions on X_test (from df_encoded)
# plot for CatBoost native model uses y_test_native and its predictions on X_test_native (from df_normal)

# For visualization consistency, we should use the same y_test_original for all.
# Let's adjust predictions of lgb_raw_model, lgb_log_model, cat_raw_model, cat_log_model on X_test_native if possible.
# This requires `X_test_native` to have the same columns as `X_test_lgb`/`X_test`.
# Since `df_normal` now contains engineered features and non-encoded categoricals, `X_test_native` is structurally different
# from the `X_test` (from `df_encoded`) that the previous models were trained on.

# For the sake of *this current step*, let's focus on adding `catboost_native_model` to the visualization.
# We'll use the respective `X_test` and `y_test` for each model type. This implies that direct visual comparison
# across models might be for different feature sets, which is an important nuance to report.

predictions = [
    (y_pred_lgb_raw_test, 'LightGBM (Raw)', lgb_raw_test_r2, '#3498db', y_test, X_test_lgb),
    (y_pred_lgb_log_test, 'LightGBM (Log)', lgb_log_test_r2, '#2ecc71', y_test, X_test_lgb),
    (y_pred_cat_raw_test, 'CatBoost (Raw)', cat_raw_test_r2, '#e74c3c', y_test, X_test),
    (y_pred_cat_log_test, 'CatBoost (Log)', cat_log_test_r2, '#f39c12', y_test, X_test),
    (y_pred_cat_native_test, 'CatBoost (Native)', cat_native_test_r2, '#9b59b6', y_test_native, X_test_native) # New model
]

# Flatten the axes array for easier iteration if it's 2D
axes = axes.flatten()

for idx, (y_pred, title, r2, color, actual_y, X_test_data) in enumerate(predictions):
    if idx >= len(axes): # Stop if we run out of subplots
        break
    ax = axes[idx]
    
    # Re-sample for visualization within the current X_test_data/actual_y
    # Ensure indices match between actual_y and y_pred
    current_sample_size = min(10000, len(actual_y))
    current_sample_idx = np.random.choice(len(actual_y), current_sample_size, replace=False)

    ax.scatter(actual_y.iloc[current_sample_idx], y_pred[current_sample_idx], alpha=0.4, s=15, color=color)
    ax.plot([actual_y.min(), actual_y.max()], [actual_y.min(), actual_y.max()],
            'r--', lw=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Length of Stay', fontsize=11)
    ax.set_ylabel('Predicted Length of Stay', fontsize=11)
    ax.set_title(f'{title}: Predicted vs Actual (R²={r2:.4f})',
                fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

# Turn off any unused subplots
for i in range(len(predictions), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('plots/final_04_predicted_vs_actual_all_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_04_predicted_vs_actual_all_v2.png")
plt.close()

# 5. Residual Analysis - All Models
fig, axes = plt.subplots(len(predictions), 2, figsize=(20, 5 * len(predictions))) # Dynamic rows

residuals = [
    (y_test - y_pred_lgb_raw_test, 'LightGBM (Raw)', '#3498db', y_pred_lgb_raw_test),
    (y_test - y_pred_lgb_log_test, 'LightGBM (Log)', '#2ecc71', y_pred_lgb_log_test),
    (y_test - y_pred_cat_raw_test, 'CatBoost (Raw)', '#e74c3c', y_pred_cat_raw_test),
    (y_test - y_pred_cat_log_test, 'CatBoost (Log)', '#f39c12', y_pred_cat_log_test),
    (y_test_native - y_pred_cat_native_test, 'CatBoost (Native)', '#9b59b6', y_pred_cat_native_test)
]

for idx, (resid, title, color, y_pred_values) in enumerate(residuals):
    current_sample_size = min(10000, len(resid))
    current_sample_idx = np.random.choice(len(resid), current_sample_size, replace=False)

    # Scatter plot
    ax_scatter = axes[idx, 0]
    ax_scatter.scatter(y_pred_values[current_sample_idx], resid.iloc[current_sample_idx],
               alpha=0.4, s=10, color=color)
    ax_scatter.axhline(y=0, color='red', linestyle='--', lw=2)
    ax_scatter.set_xlabel('Predicted Values', fontsize=10)
    ax_scatter.set_ylabel('Residuals', fontsize=10)
    ax_scatter.set_title(f'{title}: Residual Plot', fontsize=11, fontweight='bold')
    ax_scatter.grid(True, alpha=0.3)

    # Histogram
    ax_hist = axes[idx, 1]
    ax_hist.hist(resid, bins=50, edgecolor='black', alpha=0.7, color=color)
    ax_hist.axvline(x=0, color='red', linestyle='--', lw=2)
    ax_hist.set_xlabel('Residuals', fontsize=10)
    ax_hist.set_ylabel('Frequency', fontsize=10)
    ax_hist.set_title(f'{title}: Residual Distribution', fontsize=11, fontweight='bold')
    ax_hist.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('plots/final_05_residual_analysis_all_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_05_residual_analysis_all_v2.png")
plt.close()

# 6. Feature Importance Comparison
fig, axes = plt.subplots(3, 2, figsize=(18, 20)) # Changed to 3 rows

models_fi = [
    (lgb_model_raw, X_train_lgb.columns, 'LightGBM (Raw)', '#3498db'),
    (lgb_model_log, X_train_lgb.columns, 'LightGBM (Log)', '#2ecc71'),
    (cat_model_raw, X_train.columns, 'CatBoost (Raw)', '#e74c3c'),
    (cat_model_log, X_train.columns, 'CatBoost (Log)', '#f39c12'),
    (catboost_native_model, X_train_native.columns, 'CatBoost (Native)', '#9b59b6') # New model
]

axes = axes.flatten()

for idx, (model, columns, title, color) in enumerate(models_fi):
    if idx >= len(axes):
        break
    ax = axes[idx]

    importance_df = pd.DataFrame({
        'feature': columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False).head(15)

    ax.barh(range(len(importance_df)), importance_df['importance'], alpha=0.8, color=color)
    ax.set_yticks(range(len(importance_df)))
    ax.set_yticklabels(importance_df['feature'], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Importance', fontsize=11)
    ax.set_title(f'{title}: Top 15 Features', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

for i in range(len(models_fi), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.savefig('plots/final_06_feature_importance_all_v2.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_06_feature_importance_all_v2.png")
plt.close()

# 7. Raw vs Log Comparison (Existing) and adding a chart for CatBoost vs CatBoost Native
fig, axes = plt.subplots(1, 3, figsize=(20, 7)) # Increased to 3 subplots

# LightGBM comparison (Existing)
lgb_comparison = ['Raw Target', 'Log Target']
lgb_rmse = [lgb_raw_test_rmse, lgb_log_test_rmse]
lgb_colors = ['#3498db', '#2ecc71']

bars1 = axes[0].bar(lgb_comparison, lgb_rmse, color=lgb_colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[0].set_ylabel('Test RMSE', fontsize=12)
axes[0].set_title('LightGBM: Raw vs Log Target', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars1, lgb_rmse):
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

improvement_lgb = ((lgb_raw_test_rmse - lgb_log_test_rmse) / lgb_raw_test_rmse * 100)
axes[0].text(0.5, max(lgb_rmse) * 0.9, f'Change: {improvement_lgb:+.2f}%',
            ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# CatBoost (Encoded) comparison (Existing)
cat_comparison = ['Raw Target', 'Log Target']
cat_rmse = [cat_raw_test_rmse, cat_log_test_rmse]
cat_colors = ['#e74c3c', '#f39c12']

bars2 = axes[1].bar(cat_comparison, cat_rmse, color=cat_colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[1].set_ylabel('Test RMSE', fontsize=12)
axes[1].set_title('CatBoost (Encoded): Raw vs Log Target', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars2, cat_rmse):
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
improvement_cat = ((cat_raw_test_rmse - cat_log_test_rmse) / cat_raw_test_rmse * 100)
axes[1].text(0.5, max(cat_rmse) * 0.9, f'Change: {improvement_cat:+.2f}%',
             ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# New subplot: CatBoost Encoded vs CatBoost Native
cat_type_comparison = ['CatBoost (Encoded Raw)', 'CatBoost (Native)']
cat_type_rmse = [cat_raw_test_rmse, cat_native_test_rmse]
cat_type_colors = ['#e74c3c', '#9b59b6']

bars3 = axes[2].bar(cat_type_comparison, cat_type_rmse, color=cat_type_colors, alpha=0.8, edgecolor='black', linewidth=2)
axes[2].set_ylabel('Test RMSE', fontsize=12)
axes[2].set_title('CatBoost: Encoded vs Native (Raw Target)', fontsize=14, fontweight='bold')
axes[2].grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars3, cat_type_rmse):
    height = bar.get_height()
    axes[2].text(bar.get_x() + bar.get_width()/2., height,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
improvement_cat_native = ((cat_raw_test_rmse - cat_native_test_rmse) / cat_raw_test_rmse * 100)
axes[2].text(0.5, max(cat_type_rmse) * 0.9, f'Change: {improvement_cat_native:+.2f}%',
             ha='center', fontsize=11, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))


plt.tight_layout()
plt.savefig('plots/final_07_raw_vs_log_and_native.png', dpi=300, bbox_inches='tight')
print("✓ Saved: plots/final_07_raw_vs_log_and_native.png")
plt.close()

# --- FINAL SUMMARY REPORT ---
print("\n" + "="*70)
print("🏆 FINAL MODEL RANKING (by Test RMSE)")
print("="*70)
ranking = results_summary.sort_values('Test RMSE')[['Model', 'Test RMSE', 'Test MAE', 'Test R²']]
for idx, row in ranking.iterrows():
  rank = list(ranking.index).index(idx) + 1
  medal = ['🥇', '🥈', '🥉', '4️⃣', '5️⃣'][rank-1] # Updated for 5 models
  print(f"{medal} {row['Model']}: RMSE={row['Test RMSE']:.4f}, MAE={row['Test MAE']:.4f}, R²={row['Test R²']:.4f}")
best_model = ranking.iloc[0]['Model']
best_rmse = ranking.iloc[0]['Test RMSE']
print(f"\n🎯 BEST OVERALL MODEL: {best_model}")
print(f"   Test RMSE: {best_rmse:.4f}")

# Save final summary report
summary_text = f"""
FINAL MODEL COMPARISON REPORT
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
DATASET INFORMATION:

Training samples: {X_train_native.shape[0]:,} (for CatBoost Native, which uses df_normal)
Test samples: {X_test_native.shape[0]:,} (for CatBoost Native, which uses df_normal)
Features: {X_train_native.shape[1]} (for CatBoost Native, which uses df_normal)
Note: LightGBM and original CatBoost models were trained on df_encoded, which has {X_train.shape[1]} features.
Target: Length of Stay (days)

MODEL PERFORMANCE SUMMARY:
{results_summary.to_string(index=False)}

RANKING (by Test RMSE):
{ranking.to_string(index=False)}

BEST OVERALL MODEL: {best_model}
   Test RMSE: {best_rmse:.4f}
   Test MAE: {ranking.iloc[0]['Test MAE']:.4f}
   Test R²: {ranking.iloc[0]['Test R²']:.4f}

OVERFITTING ANALYSIS:
LightGBM (Raw): RMSE Gap = {lgb_raw_test_rmse - lgb_raw_train_rmse:.4f}, R² Gap = {lgb_raw_train_r2 - lgb_raw_test_r2:.4f}
LightGBM (Log): RMSE Gap = {lgb_log_test_rmse - lgb_log_train_rmse:.4f}, R² Gap = {lgb_log_train_r2 - lgb_log_test_r2:.4f}
CatBoost (Raw): RMSE Gap = {cat_raw_test_rmse - cat_raw_train_rmse:.4f}, R² Gap = {cat_raw_train_r2 - cat_raw_test_r2:.4f}
CatBoost (Log): RMSE Gap = {cat_log_test_rmse - cat_log_train_rmse:.4f}, R² Gap = {cat_log_train_r2 - cat_log_test_r2:.4f}
CatBoost (Native): RMSE Gap = {cat_native_test_rmse - cat_native_train_rmse:.4f}, R² Gap = {cat_native_train_r2 - cat_native_test_r2:.4f}

RAW vs LOG TARGET COMPARISON:
LightGBM: {'Log better' if lgb_log_test_rmse < lgb_raw_test_rmse else 'Raw better'} (Δ = {abs(lgb_raw_test_rmse - lgb_log_test_rmse):.4f} RMSE)
CatBoost (Encoded): {'Log better' if cat_log_test_rmse < cat_raw_test_rmse else 'Raw better'} (Δ = {abs(cat_raw_test_rmse - cat_log_test_rmse):.4f} RMSE)
CatBoost (Native): uses raw target by default.

CATBOOST ENCODED vs NATIVE COMPARISON:
{'Native better' if cat_native_test_rmse < cat_raw_test_rmse else 'Encoded better'} (Δ = {abs(cat_raw_test_rmse - cat_native_test_rmse):.4f} RMSE)

SAVED MODELS:
- models/lgb_raw_final.pkl
- models/lgb_log_final.pkl
- models/cat_raw_final.pkl
- models/cat_log_final.pkl
- models/catboost_native_model.pkl

SAVED VISUALIZATIONS:
- plots/final_01_overall_comparison_v2.png
- plots/final_02_overfitting_check_v2.png
- plots/final_03_metrics_heatmap_v2.png
- plots/final_04_predicted_vs_actual_all_v2.png
- plots/final_05_residual_analysis_all_v2.png
- plots/final_06_feature_importance_all_v2.png
- plots/final_07_raw_vs_log_and_native.png

================================================================================
"""
with open('models/FINAL_REPORT.txt', 'w') as f:
  f.write(summary_text)
print("\n✓ Saved: models/FINAL_REPORT.txt")
print("\n" + "="*70)
print("🎉 ALL MODELS TRAINED AND VISUALIZATIONS COMPLETE!")
print("="*70)
```

## Train CatBoost with Native Categorical Features

### Subtask:
Train the CatBoost Regressor model using `df_normal` (including newly engineered features) and native handling for categorical features, then evaluate and record its performance metrics.


**Reasoning**:
The subtask requires training a CatBoost Regressor using `df_normal` with newly engineered features and native categorical handling. The previous code block for feature engineering (cell `h2f0xfA7OtDu`) modified `df_normal`. I need to ensure the `X` and `y` are correctly extracted from this `df_normal`, categorical features are properly identified for CatBoost's native handling, and then the model is trained, evaluated, and saved as instructed.



In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
import joblib

# Ensure df_normal is used after feature engineering
X_fe = df_normal.drop('length_of_stay', axis=1)
y_fe = df_normal['length_of_stay']

# Clean column names for consistency across models
X_fe.columns = (X_fe.columns
             .str.replace('/', '_', regex=False)
             .str.replace(' ', '_', regex=False)
             .str.replace('[', '_', regex=False)
             .str.replace(']', '_', regex=False)
             .str.replace('{', '_', regex=False)
             .str.replace('}', '_', regex=False)
             .str.replace(':', '_', regex=False)
             .str.replace('"', '', regex=False)
             .str.replace("'", '', regex=False)
             .str.replace(',', '_', regex=False)
             .str.replace('-', '_', regex=False)
             .str.replace('.', '_', regex=False)
)

# Train-test split (using the X_fe and y_fe derived from df_normal)
X_train_fe, X_test_fe, y_train_fe, y_test_fe = train_test_split(X_fe, y_fe, test_size=0.2, random_state=42)

print(f"Training set size: {X_train_fe.shape}")
print(f"Test set size: {X_test_fe.shape}")

# Identify categorical features from the prepared DataFrame
# CatBoost can handle string-based categorical features directly
cat_features_native = X_train_fe.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

# Get categorical column indices
# Ensure only columns present in X_train_fe are included
cat_feature_indices = [X_train_fe.columns.get_loc(col) for col in cat_features_native if col in X_train_fe.columns]

# Train CatBoost with native categorical handling
print("\n" + "="*70)
print("TRAINING CATBOOST REGRESSOR WITH NATIVE CATEGORICAL FEATURES")
print("="*70)

catboost_native_model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    cat_features=cat_feature_indices,  # Specify categorical features
    random_seed=42,
    verbose=100, # Print metrics every 100 iterations
    early_stopping_rounds=50, # Stop if validation error doesn't improve for 50 rounds
    task_type='GPU' # Use GPU if available for faster training
)

catboost_native_model.fit(
    X_train_fe, y_train_fe,
    eval_set=(X_test_fe, y_test_fe),
    verbose=100
)

# Make predictions
y_pred_native_train = catboost_native_model.predict(X_train_fe)
y_pred_native_test = catboost_native_model.predict(X_test_fe)

# Calculate evaluation metrics
native_train_rmse = np.sqrt(mean_squared_error(y_train_fe, y_pred_native_train))
native_test_rmse = np.sqrt(mean_squared_error(y_test_fe, y_pred_native_test))
native_train_mae = mean_absolute_error(y_train_fe, y_pred_native_train))
native_test_mae = mean_absolute_error(y_test_fe, y_pred_native_test))
native_train_r2 = r2_score(y_train_fe, y_pred_native_train))
native_test_r2 = r2_score(y_test_fe, y_pred_native_test))

print("\n" + "="*70)
print("CATBOOST NATIVE MODEL PERFORMANCE")
print("="*70)
print(f"Train RMSE: {native_train_rmse:.4f} | Test RMSE: {native_test_rmse:.4f}")
print(f"Train MAE: {native_train_mae:.4f} | Test MAE: {native_test_mae:.4f}")
print(f"Train R²: {native_train_r2:.4f} | Test R²: {native_test_r2:.4f}")

# Save the trained model
joblib.dump(catboost_native_model, 'models/catboost_native_model.pkl')
print("✓ CatBoost native model saved to 'models/catboost_native_model.pkl'")

SyntaxError: unmatched ')' (ipython-input-1245506201.py, line 71)